In [ ]:
# Install the two external packages needed in Colab:
# - pydicom for reading DICOM medical imaging files
# - pandas for storing and manipulating metadata tables
!pip -q install pydicom pandas

# Mount Google Drive so the notebook can access the clinical DICOM dataset
from google.colab import drive
drive.mount('/content/drive')

# Standard Python utilities
import math
import random
from pathlib import Path

# Numerical/data handling libraries
import numpy as np
import pandas as pd
import pydicom

# PyTorch libraries for tensors, neural networks, loss functions, and 3D operations
import torch
import torch.nn as nn
import torch.nn.functional as F

# Plotting library for visualising slices, heatmaps, losses, and reformatted outputs
import matplotlib.pyplot as plt

In [ ]:
# =========================
# CONFIG
# =========================

# Root folder containing the patient folders on Google Drive
DATA_ROOT = "/content/drive/MyDrive/your_dataset_folder"

# All LGE volumes and GT heatmaps are resampled to this common shape.
# Format is (D,H,W) = (depth/slices, height, width).
TARGET_SHAPE = (56, 120, 144)   # (D,H,W)

# Random seed for reproducibility
SEED = 0

# Batch size is kept at 1, which is common for 3D medical volumes due to GPU memory limits
BATCH_SIZE = 1

# FINAL training
# Maximum number of epochs for the final selected model
NUM_EPOCHS_FINAL = 100

# Early stopping patience for final training
PATIENCE_FINAL = 18

# Weight decay for AdamW regularisation
WEIGHT_DECAY = 1e-4

# FAST CV
# Whether to run cross-validation before final training
DO_CV = True

# Number of folds for cross-validation
N_FOLDS = 2

# Maximum epochs per CV fold
CV_EPOCHS = 35

# Early stopping patience during CV
CV_PATIENCE = 6

# Hyperparameter settings to compare during CV.
# base controls the UNet channel width; lr is the learning rate.
CV_CONFIGS = [
    {"base": 10, "lr": 8e-4},
    {"base": 12, "lr": 8e-4},
]

# Whether to apply augmentation during training
AUGMENT_TRAIN = True

# Whether augmentation is allowed to include flips
APPLY_FLIPS = False

# GT plane target
# Thickness controls the binary slab width before Gaussian smoothing
GT_THICKNESS = 0.8

# Sigma controls the Gaussian blur applied to the plane slab
GT_SIGMA = 0.6

# Truncation controls Gaussian kernel size
GT_TRUNCATE = 3.0

# Number of high-probability voxels used when estimating the plane by PCA
TOPK_ESTIMATE = 30000

# -------------------------
# FIXED-CENTRE FORMULATION
# -------------------------
# The predicted/GT plane is assumed to pass through the fixed LGE volume centre.
# The network therefore mainly learns the plane orientation rather than a free centre point.
USE_FIXED_CENTER = True

# -------------------------
# REFORMATTING CONFIG
# -------------------------
# exact cine-sized canvas + thick-slice averaging

# Number of samples taken through the slice thickness when producing thick-slice reformats
REFORMAT_NUM_SLAB_SAMPLES = 9

# Weighting profile for thick-slice averaging
REFORMAT_SLICE_PROFILE = "triangular"   # "rect", "triangular", "gaussian"

# -------------------------
# LOSS WEIGHTS
# -------------------------
# Loss is a weighted combination of voxel heatmap loss terms,
# plane-normal consistency, and image-space slice supervision.
W_BCE = 0.30
W_MSE = 0.20
W_DICE = 0.20
W_GRAD = 0.10
W_NORMAL = 0.12
W_SLICE = 0.08

# lower-cost slice supervision during training
# Slice supervision is calculated on a reduced output canvas to save memory/time.
LOSS_SLICE_OUT_H = 128
LOSS_SLICE_OUT_W = 128

# Display
# Percentile windowing used for visualising clinical images
WINDOW_PERCENTILES = (1, 99)

# Use GPU if available; otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Set random seeds for reproducible splits, augmentations, and training behaviour
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Print basic runtime configuration
print("Using device:", device)
print("TARGET_SHAPE:", TARGET_SHAPE)

In [ ]:
# =========================
# BASIC HELPERS
# =========================

# Safely read a DICOM attribute.
# If the attribute does not exist, return the provided default.
def safe_get(ds, name, default=None):
    return getattr(ds, name, default)

# Convert a DICOM list-like field into a Python list of floats.
# Returns None if conversion fails.
def as_float_list(x):
    if x is None:
        return None
    try:
        return [float(v) for v in x]
    except Exception:
        return None

# Convert input into a detached float32 torch tensor on the selected device.
# If input is already a tensor, clone it to avoid modifying the original.
def to_torch_float(x, device="cpu"):
    if torch.is_tensor(x):
        return x.detach().clone().to(device=device, dtype=torch.float32)
    return torch.tensor(x, dtype=torch.float32, device=device)

# Compute the plane normal from DICOM ImageOrientationPatient.
# IOP contains row direction followed by column direction in patient coordinates.
# The slice normal is row x column.
def compute_normal_from_iop(iop):
    if iop is None or len(iop) != 6:
        return None
    row = np.array(iop[:3], dtype=float)
    col = np.array(iop[3:], dtype=float)
    n = np.cross(row, col)
    nrm = np.linalg.norm(n)
    if nrm < 1e-8:
        return None
    return n / nrm

# Round values into a tuple so they can be used as stable grouping keys.
# This is useful because DICOM coordinates may contain tiny floating-point differences.
def rounded_tuple(vals, ndigits=3):
    if vals is None:
        return None
    return tuple(round(float(v), ndigits) for v in vals)

# Min-max normalise a tensor to approximately [0,1].
def normalize_img_torch(x, eps=1e-8):
    x = x.float()
    x = x - x.min()
    x = x / (x.max() + eps)
    return x

# Return the voxel-space centre point of a 3D volume.
# Format is (z,y,x).
def voxel_center_from_shape(shape):
    D, H, W = shape
    return torch.tensor([(D - 1) / 2.0, (H - 1) / 2.0, (W - 1) / 2.0], dtype=torch.float32)

# Compute sign-invariant angle between two vectors.
# abs(dot) is used because plane normals n and -n represent the same plane.
def angle_between(u, v, eps=1e-8):
    u = u / (u.norm() + eps)
    v = v / (v.norm() + eps)
    c = torch.clamp(torch.abs((u * v).sum()), 0.0, 1.0)
    return float(torch.rad2deg(torch.acos(c)))

# Compute cosine similarity between two 2D images after flattening.
def cosine_similarity_2d(a, b, eps=1e-8):
    a = a.float().reshape(-1)
    b = b.float().reshape(-1)
    return float(torch.dot(a, b) / (a.norm() * b.norm() + eps))

# Euclidean point error in voxel units.
def point_error(p_pred, p_gt):
    return float(torch.norm(p_pred - p_gt).item())

# Distance between predicted point and GT plane along the GT normal direction.
# This measures through-plane offset error.
def plane_offset_error(n_gt, p_gt, p_pred):
    n_gt = n_gt / (n_gt.norm() + 1e-8)
    return float(torch.abs(torch.dot((p_pred - p_gt), n_gt)).item())

# Percentile-window an image for display.
# This improves visual contrast for clinical intensity images.
def window_tensor(x, low_pct=1, high_pct=99, eps=1e-8):
    x_np = x.detach().cpu().numpy()
    lo = np.percentile(x_np, low_pct)
    hi = np.percentile(x_np, high_pct)
    x_np = np.clip(x_np, lo, hi)
    x_np = (x_np - lo) / (hi - lo + eps)
    return torch.tensor(x_np, dtype=torch.float32)

# Rescale a voxel coordinate from the original image shape into the resized target shape.
def rescale_voxel_point(point_zyx, old_shape, new_shape):
    point_zyx = np.array(point_zyx, dtype=float)
    old_shape = np.array(old_shape, dtype=float)
    new_shape = np.array(new_shape, dtype=float)
    scale = (new_shape - 1.0) / np.maximum(old_shape - 1.0, 1.0)
    return point_zyx * scale

# Rescale a voxel-space displacement vector when the volume is resized.
# If normalize=True, the output is converted back to a unit vector.
def rescale_voxel_displacement(vec_zyx, old_shape, new_shape, normalize=False, eps=1e-8):
    vec_zyx = np.array(vec_zyx, dtype=float)
    old_shape = np.array(old_shape, dtype=float)
    new_shape = np.array(new_shape, dtype=float)
    scale = (new_shape - 1.0) / np.maximum(old_shape - 1.0, 1.0)
    out = vec_zyx * scale
    if normalize:
        n = np.linalg.norm(out)
        if n > eps:
            out = out / n
    return out

In [ ]:
# =========================
# HEATMAP + PLANE HELPERS
# =========================

# Build a separable 3D Gaussian kernel.
# This is used to blur the binary plane slab into a smooth heatmap target.
def gaussian_kernel_3d(sigma: float, truncate: float = 3.0, device="cpu"):
    if sigma <= 0:
        raise ValueError("sigma must be > 0")

    # Kernel radius is based on sigma and truncate.
    radius = int(truncate * sigma + 0.5)
    coords = torch.arange(-radius, radius + 1, device=device).float()

    # 1D Gaussian
    g1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g1d = g1d / g1d.sum()

    # Outer product to form 3D Gaussian
    g3d = g1d[:, None, None] * g1d[None, :, None] * g1d[None, None, :]
    g3d = g3d / g3d.sum()

    # Return in Conv3D format: (out_channels, in_channels, D, H, W)
    return g3d.view(1, 1, *g3d.shape)


# Create a 3D plane heatmap directly from a shape, normal, and point.
# The heatmap is made by:
# 1. computing distance of every voxel to the plane,
# 2. thresholding into a slab,
# 3. Gaussian blurring,
# 4. normalising to [0,1].
def plane_heatmap_from_shape(
    shape,
    normal_zyx,
    point_on_plane_zyx,
    thickness=1.0,
    sigma=0.8,
    truncate=3.0,
    device="cpu"
):
    D, H, W = shape

    # Convert normal and point to tensors
    normal = to_torch_float(normal_zyx, device=device)
    point = to_torch_float(point_on_plane_zyx, device=device)

    # Ensure normal is unit length
    normal = normal / (normal.norm() + 1e-8)

    # Create coordinate grid in voxel coordinates
    z = torch.arange(D, device=device).float()
    y = torch.arange(H, device=device).float()
    x = torch.arange(W, device=device).float()
    zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")

    # coords has shape (3,D,H,W), using (z,y,x) ordering
    coords = torch.stack([zz, yy, xx], dim=0)

    # Vector from plane point to every voxel
    v = coords - point.view(3, 1, 1, 1)

    # Signed distance to plane, then absolute distance
    signed_dist = (v * normal.view(3, 1, 1, 1)).sum(dim=0)
    dist = signed_dist.abs()

    # Binary slab around the plane
    mask = (dist <= thickness).float()

    # Blur the slab with a 3D Gaussian kernel
    kernel = gaussian_kernel_3d(sigma=sigma, truncate=truncate, device=device)
    mask_5d = mask.unsqueeze(0).unsqueeze(0)

    # Replicate padding prevents artificial edge loss during convolution
    kD, kH, kW = kernel.shape[-3:]
    pad = (kW // 2, kW // 2, kH // 2, kH // 2, kD // 2, kD // 2)
    mask_padded = F.pad(mask_5d, pad, mode="replicate")

    # Apply convolution and remove batch/channel dimensions
    heat = F.conv3d(mask_padded, kernel)[0, 0]

    # Normalise the heatmap for stable learning
    heat = heat / (heat.max().clamp_min(1e-8))
    return heat


# Estimate plane normal and centre point from a heatmap using weighted PCA.
# Bright heatmap voxels are treated as weighted samples from the plane.
def estimate_plane_from_heatmap(heatmap_zyx, topk=30000, eps=1e-8):
    assert heatmap_zyx.ndim == 3
    D, H, W = heatmap_zyx.shape

    # Flatten heatmap into voxel weights
    w = heatmap_zyx.reshape(-1)

    # Optionally use only the top-k brightest voxels to focus PCA on the plane region
    if topk is not None and topk < w.numel():
        vals, idx = torch.topk(w, k=topk)
        w = vals.clamp_min(0.0)

        # Convert flat indices back into z,y,x voxel coordinates
        z = (idx // (H * W)).float()
        y = ((idx % (H * W)) // W).float()
        x = (idx % W).float()
        coords = torch.stack([z, y, x], dim=1)
    else:
        # Use all voxels if topk is None or larger than the volume size
        z = torch.arange(D, device=heatmap_zyx.device)
        y = torch.arange(H, device=heatmap_zyx.device)
        x = torch.arange(W, device=heatmap_zyx.device)
        zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")
        coords = torch.stack([zz, yy, xx], dim=-1).reshape(-1, 3).float()
        w = w.clamp_min(0.0)

    # Normalise weights so they sum to 1
    w = w / (w.sum() + eps)

    # Weighted centroid gives estimated point on the plane
    point = (w[:, None] * coords).sum(dim=0)

    # Weighted covariance of voxel coordinates around the centroid
    Xc = coords - point[None, :]
    C = (w[:, None, None] * (Xc[:, :, None] * Xc[:, None, :])).sum(dim=0)

    # The eigenvector with the smallest eigenvalue is the plane normal
    evals, evecs = torch.linalg.eigh(C)
    normal = evecs[:, 0]
    normal = normal / (normal.norm() + eps)
    return normal, point

In [ ]:
# =========================
# DICOM LOADERS
# =========================

# Load an enhanced multi-frame LGE DICOM file and extract both image data and geometry.
def load_lge_multiframe(lge_path):
    # Read the full DICOM file including pixel data
    ds = pydicom.dcmread(str(lge_path))

    # Convert pixel array to float32 volume
    vol = ds.pixel_array.astype(np.float32)

    # Enhanced multi-frame DICOMs should contain per-frame geometry
    if not hasattr(ds, "PerFrameFunctionalGroupsSequence"):
        raise ValueError(f"LGE file does not contain PerFrameFunctionalGroupsSequence: {lge_path}")

    pfg = ds.PerFrameFunctionalGroupsSequence
    frame0 = pfg[0]

    # Read image orientation from the first frame
    if hasattr(frame0, "PlaneOrientationSequence"):
        iop = np.array(frame0.PlaneOrientationSequence[0].ImageOrientationPatient, dtype=float)
    else:
        raise ValueError(f"Could not find PlaneOrientationSequence in first LGE frame: {lge_path}")

    # DICOM orientation gives row and column directions in patient coordinates
    row_dir = iop[:3]
    col_dir = iop[3:]

    # Slice direction is the cross product of row and column directions
    slice_dir = np.cross(row_dir, col_dir)
    slice_dir = slice_dir / np.linalg.norm(slice_dir)

    # Read first-frame image position as the LGE origin in patient space
    if hasattr(frame0, "PlanePositionSequence"):
        ipp0 = np.array(frame0.PlanePositionSequence[0].ImagePositionPatient, dtype=float)
    else:
        raise ValueError(f"Could not find PlanePositionSequence in first LGE frame: {lge_path}")

    ps = None
    dz = None

    # Try to read pixel spacing and slice spacing from shared functional groups
    if hasattr(ds, "SharedFunctionalGroupsSequence"):
        shared = ds.SharedFunctionalGroupsSequence[0]
        if hasattr(shared, "PixelMeasuresSequence"):
            pm = shared.PixelMeasuresSequence[0]
            if hasattr(pm, "PixelSpacing"):
                ps = np.array(pm.PixelSpacing, dtype=float)
            if hasattr(pm, "SpacingBetweenSlices"):
                dz = float(pm.SpacingBetweenSlices)
            elif hasattr(pm, "SliceThickness"):
                dz = float(pm.SliceThickness)

    # If spacing was not found in shared metadata, try per-frame metadata
    if (ps is None or dz is None) and hasattr(frame0, "PixelMeasuresSequence"):
        pm = frame0.PixelMeasuresSequence[0]
        if ps is None and hasattr(pm, "PixelSpacing"):
            ps = np.array(pm.PixelSpacing, dtype=float)
        if dz is None:
            if hasattr(pm, "SpacingBetweenSlices"):
                dz = float(pm.SpacingBetweenSlices)
            elif hasattr(pm, "SliceThickness"):
                dz = float(pm.SliceThickness)

    # Fall back to top-level PixelSpacing if available
    if ps is None and hasattr(ds, "PixelSpacing"):
        ps = np.array(ds.PixelSpacing, dtype=float)

    # If slice spacing is still missing, estimate it from consecutive frame positions
    if dz is None:
        positions = []
        for fr in pfg:
            ipp = np.array(fr.PlanePositionSequence[0].ImagePositionPatient, dtype=float)
            positions.append(ipp)
        positions = np.stack(positions, axis=0)

        # Project frame-to-frame differences onto slice direction
        diffs = positions[1:] - positions[:-1]
        dzs = np.abs(diffs @ slice_dir)
        dz = float(np.median(dzs))

    # Fail if required spacing information is still missing
    if ps is None:
        raise ValueError(f"Could not determine PixelSpacing for LGE: {lge_path}")
    if dz is None:
        raise ValueError(f"Could not determine slice spacing for LGE: {lge_path}")

    # Return both image volume and geometry needed for patient-to-voxel transforms
    return {
        "ds": ds,
        "volume": vol,
        "shape": vol.shape,
        "origin_patient": ipp0,
        "row_dir": row_dir,
        "col_dir": col_dir,
        "slice_dir": slice_dir,
        "spacing_row": float(ps[0]),
        "spacing_col": float(ps[1]),
        "spacing_slice": float(dz),
    }


# Load cine DICOM metadata from a patient cine folder.
# This reads headers only, not pixels, so it is faster.
def load_cine_metadata(cine_root):
    files = sorted([p for p in Path(cine_root).rglob("*") if p.is_file()])
    rows = []

    for fp in files:
        # stop_before_pixels=True avoids loading image data at this stage
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)

        iop = as_float_list(safe_get(ds, "ImageOrientationPatient"))
        ipp = as_float_list(safe_get(ds, "ImagePositionPatient"))
        normal = compute_normal_from_iop(iop)

        # Store relevant metadata for grouping slices and selecting cardiac frame
        rows.append({
            "path": str(fp),
            "filename": fp.name,
            "InstanceNumber": safe_get(ds, "InstanceNumber"),
            "TriggerTime": safe_get(ds, "TriggerTime"),
            "ImageOrientationPatient": iop,
            "ImagePositionPatient": ipp,
            "PlaneNormal": normal.tolist() if normal is not None else None,
            "Rows": int(safe_get(ds, "Rows")),
            "Columns": int(safe_get(ds, "Columns")),
            "PixelSpacing": as_float_list(safe_get(ds, "PixelSpacing")),
            "SliceThickness": safe_get(ds, "SliceThickness"),
            "SpacingBetweenSlices": safe_get(ds, "SpacingBetweenSlices"),
        })

    df = pd.DataFrame(rows)

    # Create grouping keys for slice position and orientation
    df["IPP_key"] = df["ImagePositionPatient"].apply(lambda x: rounded_tuple(x, 3))
    df["normal_key"] = df["PlaneNormal"].apply(lambda x: rounded_tuple(x, 6) if x is not None else None)

    # Project each image position onto its normal to determine stack order
    def proj(row):
        if row["ImagePositionPatient"] is None or row["PlaneNormal"] is None:
            return None
        p = np.array(row["ImagePositionPatient"], dtype=float)
        n = np.array(row["PlaneNormal"], dtype=float)
        return float(np.dot(p, n))

    df["slice_proj"] = df.apply(proj, axis=1)

    # Group frames by unique slice position/orientation
    slice_groups = (
        df.groupby(["IPP_key", "normal_key"], dropna=False)
          .agg(
              n_files=("filename", "count"),
              proj_mean=("slice_proj", "mean"),
              first_filename=("filename", "first"),
              first_path=("path", "first"),
          )
          .reset_index()
          .sort_values("proj_mean")
          .reset_index(drop=True)
    )

    # Assign ordered slice index along the cine stack
    slice_groups["slice_index_along_stack"] = np.arange(len(slice_groups))
    return df, slice_groups


# Choose a representative cardiac phase using TriggerTime.
# If non-zero trigger times exist, use the middle non-zero time.
def choose_reference_trigger_time(cine_df):
    times = sorted([float(t) for t in cine_df["TriggerTime"].dropna().unique()])
    nonzero = [t for t in times if abs(t) > 1e-6]
    times_use = nonzero if len(nonzero) > 0 else times
    return float(times_use[len(times_use) // 2])


# Build one cine short-axis stack at the chosen representative trigger time.
# For each slice position, select the frame closest to the reference trigger time.
def build_true_cine_stack_for_trigger(cine_df, cine_slice_groups, ref_trigger_time):
    chosen_rows = []

    for _, sg in cine_slice_groups.iterrows():
        sub = cine_df[
            (cine_df["IPP_key"] == sg["IPP_key"]) &
            (cine_df["normal_key"] == sg["normal_key"])
        ].copy()

        sub = sub[sub["TriggerTime"].notna()].copy()
        if len(sub) == 0:
            continue

        # Pick the cine frame closest to the selected cardiac phase
        sub["trigger_diff"] = np.abs(sub["TriggerTime"].astype(float) - float(ref_trigger_time))
        best = sub.sort_values("trigger_diff").iloc[0].copy()
        best["slice_index_along_stack"] = int(sg["slice_index_along_stack"])
        chosen_rows.append(best)

    if len(chosen_rows) == 0:
        raise ValueError("Could not build cine stack at representative trigger time.")

    # Return ordered cine stack metadata
    out = pd.DataFrame(chosen_rows).sort_values("slice_index_along_stack").reset_index(drop=True)
    return out


# Load the actual cine pixel images for the selected cine stack.
def load_cine_stack_pixels(gt_stack_df):
    imgs = []
    for _, row in gt_stack_df.iterrows():
        ds = pydicom.dcmread(row["path"])
        img = ds.pixel_array.astype(np.float32)

        # Normalise each cine slice independently to [0,1]
        img = img - img.min()
        img = img / (img.max() + 1e-8)
        imgs.append(img)
    return imgs# =========================
# PATIENT/LGE VOXEL TRANSFORMS
# =========================

# Convert a patient-space coordinate, in DICOM physical coordinates,
# into LGE voxel coordinates in (z,y,x) order.
def patient_to_lge_voxel(patient_xyz, lge):
    origin = lge["origin_patient"]
    row_dir = lge["row_dir"]
    col_dir = lge["col_dir"]
    slice_dir = lge["slice_dir"]

    dr = lge["spacing_row"]
    dc = lge["spacing_col"]
    dz = lge["spacing_slice"]

    # Patient-space displacement from LGE origin
    v = np.array(patient_xyz, dtype=float) - origin

    # Project displacement onto LGE row, column, and slice axes,
    # then divide by spacing to convert mm into voxel units.
    i = np.dot(v, row_dir) / dr
    j = np.dot(v, col_dir) / dc
    k = np.dot(v, slice_dir) / dz

    # Return in volume indexing order: (z,y,x)
    return np.array([k, i, j], dtype=float)   # (z,y,x)


# Convert a patient-space direction vector into a unit vector in LGE voxel space.
# This is used for plane normals.
def patient_vector_to_lge_voxel_vector(patient_vec, lge):
    row_dir = lge["row_dir"]
    col_dir = lge["col_dir"]
    slice_dir = lge["slice_dir"]

    dr = lge["spacing_row"]
    dc = lge["spacing_col"]
    dz = lge["spacing_slice"]

    vec = np.array(patient_vec, dtype=float)

    # Project vector onto voxel axes and account for voxel spacing
    i = np.dot(vec, row_dir) / dr
    j = np.dot(vec, col_dir) / dc
    k = np.dot(vec, slice_dir) / dz

    out = np.array([k, i, j], dtype=float)

    # Normalise because this function is intended for direction vectors
    out = out / np.linalg.norm(out)
    return out


# Convert a patient-space displacement vector into LGE voxel displacement.
# Unlike the normal-vector function, this does not normalise the result,
# so magnitude in voxel units is preserved.
def patient_vector_to_lge_voxel_displacement(patient_vec, lge):
    """
    Convert a patient-space displacement vector (mm) into LGE voxel displacement (z,y,x),
    without normalising.
    """
    row_dir = lge["row_dir"]
    col_dir = lge["col_dir"]
    slice_dir = lge["slice_dir"]

    dr = lge["spacing_row"]
    dc = lge["spacing_col"]
    dz = lge["spacing_slice"]

    vec = np.array(patient_vec, dtype=float)

    # Convert physical displacement into voxel displacement components
    i = np.dot(vec, row_dir) / dr
    j = np.dot(vec, col_dir) / dc
    k = np.dot(vec, slice_dir) / dz

    return np.array([k, i, j], dtype=float)

In [ ]:
# =========================
# BUILD ONE PATIENT SAMPLE
# - GT cine stack = actual cine SAX stack from DICOMs
# - GT heatmap = cine SAX orientation + FIXED LGE centre
# =========================

# Build one full training sample from a single patient folder.
# Output:
# - LGE volume as input
# - GT plane heatmap as target
# - metadata needed for reformatted slice/stack evaluation
def build_one_patient_sample(patient_dir):
    patient_dir = Path(patient_dir)
    cine_root = patient_dir / "cine"
    lge_root = patient_dir / "LGE"

    # Require expected folder structure
    if not cine_root.exists() or not lge_root.exists():
        raise FileNotFoundError(f"{patient_dir} must contain cine/ and LGE/")

    # Locate LGE DICOM file
    lge_files = [p for p in lge_root.rglob("*") if p.is_file()]
    if len(lge_files) == 0:
        raise FileNotFoundError(f"No LGE file found in {lge_root}")

    # Load LGE image and geometry
    lge_path = lge_files[0]
    lge = load_lge_multiframe(lge_path)

    # Load cine metadata and group frames into slice locations
    cine_df, cine_slice_groups = load_cine_metadata(cine_root)
    if len(cine_slice_groups) == 0:
        raise ValueError(f"No cine slices found in {cine_root}")

    # Build the true cine SAX stack at one representative cardiac phase
    ref_trigger_time = choose_reference_trigger_time(cine_df)
    gt_stack_df = build_true_cine_stack_for_trigger(cine_df, cine_slice_groups, ref_trigger_time)
    gt_stack_pixels = load_cine_stack_pixels(gt_stack_df)

    # Use the middle cine slice as the reference slice
    rep = gt_stack_df.iloc[len(gt_stack_df) // 2]

    # Extract cine row and column directions from DICOM orientation
    cine_iop = np.array(rep["ImageOrientationPatient"], dtype=float)
    cine_row_dir_patient = cine_iop[:3]
    cine_col_dir_patient = cine_iop[3:]

    # Cine short-axis plane normal in patient coordinates
    cine_normal_patient = np.cross(cine_row_dir_patient, cine_col_dir_patient)
    cine_normal_patient = cine_normal_patient / np.linalg.norm(cine_normal_patient)

    # Convert cine normal into LGE voxel coordinates
    cine_normal_voxel = patient_vector_to_lge_voxel_vector(cine_normal_patient, lge)

    # Read cine in-plane pixel spacing and image size
    pixel_spacing = np.array(rep["PixelSpacing"], dtype=float)
    cine_ps_row = float(pixel_spacing[0])
    cine_ps_col = float(pixel_spacing[1])

    cine_rows = int(rep["Rows"])
    cine_cols = int(rep["Columns"])

    # One-pixel in-plane displacement vectors in LGE voxel coordinates
    # These are later used to sample reformatted LGE slices on a cine-like grid.
    row_vec_vox = patient_vector_to_lge_voxel_displacement(cine_row_dir_patient * cine_ps_row, lge)
    col_vec_vox = patient_vector_to_lge_voxel_displacement(cine_col_dir_patient * cine_ps_col, lge)

    # Fixed centre in LGE voxel space
    lge_center_voxel = np.array([
        (lge["shape"][0] - 1) / 2.0,
        (lge["shape"][1] - 1) / 2.0,
        (lge["shape"][2] - 1) / 2.0
    ], dtype=float)

    # GT heatmap goes through fixed centre
    # LGE image is normalised and used as the model input.
    vol = torch.tensor(lge["volume"], dtype=torch.float32)
    vol = normalize_img_torch(vol)

    # Generate GT plane heatmap using the cine SAX normal and fixed LGE centre
    heat = plane_heatmap_from_shape(
        shape=lge["shape"],
        normal_zyx=cine_normal_voxel,
        point_on_plane_zyx=lge_center_voxel,
        thickness=GT_THICKNESS,
        sigma=GT_SIGMA,
        truncate=GT_TRUNCATE,
        device="cpu"
    ).cpu()

    # Stack offsets from true cine slice positions
    # These offsets are relative to the middle cine slice.
    proj_vals = gt_stack_df["slice_proj"].astype(float).values
    mid_idx = len(proj_vals) // 2
    offsets_mm = proj_vals - proj_vals[mid_idx]

    # Convert mm offsets along the cine normal into LGE voxel offsets
    normal_step_vox_per_mm = np.linalg.norm(
        patient_vector_to_lge_voxel_displacement(cine_normal_patient, lge)
    )

    stack_offsets_vox = offsets_mm * normal_step_vox_per_mm

    # Cine slice thickness in mm
    # If missing, fall back to spacing between slices, then to 8 mm.
    slice_thickness_mm = rep["SliceThickness"]
    if slice_thickness_mm is None:
        slice_thickness_mm = rep["SpacingBetweenSlices"]
    if slice_thickness_mm is None:
        slice_thickness_mm = 8.0
    slice_thickness_mm = float(slice_thickness_mm)

    # Store all geometry and image data required later for training/evaluation
    meta = {
        "patient_id": patient_dir.name,
        "lge_shape": lge["shape"],
        "lge_path": str(lge_path),
        "cine_root": str(cine_root),
        "ref_trigger_time": float(ref_trigger_time),

        "cine_normal_patient": cine_normal_patient.astype(np.float32),
        "cine_normal_voxel": cine_normal_voxel.astype(np.float32),

        "lge_center_voxel": lge_center_voxel.astype(np.float32),

        "cine_rows": cine_rows,
        "cine_cols": cine_cols,
        "cine_ps_row": float(cine_ps_row),
        "cine_ps_col": float(cine_ps_col),

        "row_vec_vox": row_vec_vox.astype(np.float32),
        "col_vec_vox": col_vec_vox.astype(np.float32),

        "slice_thickness_mm": float(slice_thickness_mm),
        "normal_step_vox_per_mm": float(normal_step_vox_per_mm),

        "offsets_mm": np.array(offsets_mm, dtype=np.float32),
        "stack_offsets_vox": np.array(stack_offsets_vox, dtype=np.float32),

        "gt_stack_pixels": gt_stack_pixels,
    }

    # Return input image and GT heatmap with channel dimension added
    return vol.unsqueeze(0), heat.unsqueeze(0), meta

In [ ]:
# =========================
# LOAD ALL PATIENTS
# =========================

# Find all patient folders whose names start with "p"
patient_dirs = sorted([p for p in Path(DATA_ROOT).iterdir() if p.is_dir() and p.name.startswith("p")])
print("Found patient folders:", [p.name for p in patient_dirs])

# Store successfully loaded samples and failed patients separately
samples = []
failed = []

# Try to build one sample per patient
for i, pdir in enumerate(patient_dirs, 1):
    try:
        Xp, Yp, meta = build_one_patient_sample(pdir)
        samples.append((Xp, Yp, meta))
        print(f"[{i}/{len(patient_dirs)}] Loaded {pdir.name}: X {tuple(Xp.shape)} | Y {tuple(Yp.shape)}")
    except Exception as e:
        # Keep track of failures without stopping the whole loading process
        failed.append((pdir.name, str(e)))
        print(f"[{i}/{len(patient_dirs)}] FAILED {pdir.name}: {e}")

# Print loading summary
print("\nLoaded samples:", len(samples))
print("Failed samples:", len(failed))
print("First few failures:", failed[:5])

# Stop execution if no usable patient samples were loaded
if len(samples) == 0:
    raise ValueError("No patient samples loaded.")

In [ ]:
# =========================
# RESAMPLE + BUILD GT GEOMETRY
# =========================

# Resample a single-channel 3D tensor to a target shape.
# Input shape is (1,D,H,W), output remains (1,target_D,target_H,target_W).
def resample_1ch_tensor(vol_1ch, target_shape, mode="trilinear"):
    assert vol_1ch.ndim == 4
    x = vol_1ch.unsqueeze(0)
    x = F.interpolate(x, size=target_shape, mode=mode, align_corners=False)
    return x[0]


# Create three coordinate channels in normalised coordinates [-1,1].
# These are added to the image input so the network has spatial position information.
def make_coord_channels(shape):
    D, H, W = shape
    z = torch.linspace(-1, 1, D).view(D, 1, 1).expand(D, H, W)
    y = torch.linspace(-1, 1, H).view(1, H, 1).expand(D, H, W)
    x = torch.linspace(-1, 1, W).view(1, 1, W).expand(D, H, W)
    return torch.stack([z, y, x], dim=0).float()


# Convert a list of cine 2D slices into a 3D tensor stack
def stack_list_to_tensor(gt_stack_pixels):
    arr = np.stack(gt_stack_pixels, axis=0)
    return torch.tensor(arr, dtype=torch.float32)


# Fixed coordinate channels for the target volume shape
coord_channels_fixed = make_coord_channels(TARGET_SHAPE)

# Fixed centre point in target voxel space
TARGET_CENTER = voxel_center_from_shape(TARGET_SHAPE)

processed_samples = []

# Process each loaded patient into a common shape and consistent geometry representation
for X, Y, meta in samples:
    old_shape = meta["lge_shape"]

    # Resample image and heatmap to the common target shape
    Xr = resample_1ch_tensor(X.float(), TARGET_SHAPE, mode="trilinear")
    Yr = resample_1ch_tensor(Y.float(), TARGET_SHAPE, mode="trilinear")
    Yr = torch.clamp(Yr, 0.0, 1.0)

    # Re-estimate the GT normal from the resampled heatmap
    n_gt, _ = estimate_plane_from_heatmap(Yr[0], topk=TOPK_ESTIMATE)

    # Convert the true cine stack into a tensor for later comparison
    gt_cine_stack = stack_list_to_tensor(meta["gt_stack_pixels"])

    # Rescale cine row/column voxel displacement vectors into the target volume grid
    row_vec_vox_rescaled = rescale_voxel_displacement(
        meta["row_vec_vox"], old_shape, TARGET_SHAPE, normalize=False
    )
    col_vec_vox_rescaled = rescale_voxel_displacement(
        meta["col_vec_vox"], old_shape, TARGET_SHAPE, normalize=False
    )

    # Rescale normal step size after resizing the LGE volume
    normal_step_vox_per_mm_rescaled = np.linalg.norm(
        rescale_voxel_displacement(
            np.array(meta["cine_normal_voxel"], dtype=np.float32),
            old_shape,
            TARGET_SHAPE,
            normalize=False
        )
    )

    # Convert stack offsets from mm into target-grid voxel units
    stack_offsets_vox_rescaled = np.array(meta["offsets_mm"], dtype=np.float32) * normal_step_vox_per_mm_rescaled

    # Store processed tensors and geometry for the dataset
    processed_samples.append({
        "patient_id": meta["patient_id"],
        "X_img": Xr,
        "Y_plane": Yr,
        "Y_normal": n_gt.float(),
        "Y_point_vox": TARGET_CENTER.clone().float(),   # FIXED CENTRE

        "gt_cine_stack": gt_cine_stack.float(),
        "stack_offsets_vox": torch.tensor(stack_offsets_vox_rescaled, dtype=torch.float32),

        "cine_rows": int(meta["cine_rows"]),
        "cine_cols": int(meta["cine_cols"]),
        "cine_ps_row": float(meta["cine_ps_row"]),
        "cine_ps_col": float(meta["cine_ps_col"]),

        "row_vec_vox": torch.tensor(row_vec_vox_rescaled, dtype=torch.float32),
        "col_vec_vox": torch.tensor(col_vec_vox_rescaled, dtype=torch.float32),

        "slice_thickness_mm": float(meta["slice_thickness_mm"]),
        "normal_step_vox_per_mm": float(normal_step_vox_per_mm_rescaled),
    })

# Print checks for the first processed sample
print("Processed samples:", len(processed_samples))
print("Example patient:", processed_samples[0]["patient_id"])
print("Example X_img shape:", tuple(processed_samples[0]["X_img"].shape))
print("Example Y_plane shape:", tuple(processed_samples[0]["Y_plane"].shape))
print("Example Y_normal:", processed_samples[0]["Y_normal"])
print("Example Y_point_vox:", processed_samples[0]["Y_point_vox"])
print("Example gt_cine_stack shape:", tuple(processed_samples[0]["gt_cine_stack"].shape))
print("Example stack_offsets_vox:", processed_samples[0]["stack_offsets_vox"])
print("Example cine rows/cols:", processed_samples[0]["cine_rows"], processed_samples[0]["cine_cols"])
print("Example row_vec_vox:", processed_samples[0]["row_vec_vox"])
print("Example col_vec_vox:", processed_samples[0]["col_vec_vox"])
print("Example slice_thickness_mm:", processed_samples[0]["slice_thickness_mm"])
print("Example normal_step_vox_per_mm:", processed_samples[0]["normal_step_vox_per_mm"])

In [ ]:
# =========================
# SPLIT TRAIN / VAL / TEST
# =========================

# Split patient samples into train, validation, and test sets.
# Splitting is done at patient level to avoid leakage between sets.
def split_patients(samples, train_frac=0.7, val_frac=0.15, seed=0):
    idx = list(range(len(samples)))
    rng = random.Random(seed)
    rng.shuffle(idx)

    n = len(idx)
    n_train = max(1, int(round(train_frac * n)))
    n_val = max(1, int(round(val_frac * n)))
    n_test = n - n_train - n_val

    # Ensure there is at least one test case
    if n_test < 1:
        n_test = 1
        if n_train > n_val:
            n_train -= 1
        else:
            n_val -= 1

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx


# Generate patient-level split indices
train_idx, val_idx, test_idx = split_patients(processed_samples, train_frac=0.7, val_frac=0.15, seed=SEED)

# Create split sample lists
train_samples = [processed_samples[i] for i in train_idx]
val_samples   = [processed_samples[i] for i in val_idx]
test_samples  = [processed_samples[i] for i in test_idx]

# Training + validation samples are used for cross-validation
trainval_samples = train_samples + val_samples

# Print split summary
print("Train patients:", len(train_samples))
print("Val patients:", len(val_samples))
print("Test patients:", len(test_samples))

print("Train IDs:", [s["patient_id"] for s in train_samples[:10]])
print("Val IDs:", [s["patient_id"] for s in val_samples])
print("Test IDs:", [s["patient_id"] for s in test_samples])

In [ ]:
# =========================
# AUGMENTATION
# =========================

# Build a 3D affine matrix for PyTorch affine_grid/grid_sample.
# Parameters include translation, scaling, and rotation.
def make_affine_3d(
    tx=0.0, ty=0.0, tz=0.0,
    sx=1.0, sy=1.0, sz=1.0,
    rx_deg=0.0, ry_deg=0.0, rz_deg=0.0,
    device="cpu"
):
    # Convert degrees to radians
    rx = math.radians(rx_deg)
    ry = math.radians(ry_deg)
    rz = math.radians(rz_deg)

    # Precompute sine/cosine values
    cx, sxn = math.cos(rx), math.sin(rx)
    cy, syn = math.cos(ry), math.sin(ry)
    cz, szn = math.cos(rz), math.sin(rz)

    # Rotation around x-axis
    Rx = torch.tensor([
        [1, 0, 0],
        [0, cx, -sxn],
        [0, sxn, cx],
    ], dtype=torch.float32, device=device)

    # Rotation around y-axis
    Ry = torch.tensor([
        [cy, 0, syn],
        [0, 1, 0],
        [-syn, 0, cy],
    ], dtype=torch.float32, device=device)

    # Rotation around z-axis
    Rz = torch.tensor([
        [cz, -szn, 0],
        [szn, cz, 0],
        [0, 0, 1],
    ], dtype=torch.float32, device=device)

    # Combined rotation and scaling
    R = Rz @ Ry @ Rx
    S = torch.diag(torch.tensor([sx, sy, sz], dtype=torch.float32, device=device))
    A3 = R @ S

    # PyTorch affine_grid expects a 3x4 matrix
    A = torch.zeros((3, 4), dtype=torch.float32, device=device)
    A[:, :3] = A3
    A[:, 3] = torch.tensor([tx, ty, tz], dtype=torch.float32, device=device)
    return A


# Apply the same affine transform to the input image, GT plane heatmap,
# and coordinate channels so they remain spatially aligned.
def apply_paired_affine_3d(X_img_1ch, Y_plane_1ch, coord_3ch, A):
    img5 = X_img_1ch.unsqueeze(0)
    plane5 = Y_plane_1ch.unsqueeze(0)
    coord5 = coord_3ch.unsqueeze(0)

    # Create sampling grid from affine matrix
    grid = F.affine_grid(A.unsqueeze(0), size=img5.shape, align_corners=False)

    # Bilinear interpolation for image and heatmap
    img_aug = F.grid_sample(img5, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    plane_aug = F.grid_sample(plane5, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

    # Border padding for coordinate channels keeps spatial coordinates valid near edges
    coord_aug = F.grid_sample(coord5, grid, mode="bilinear", padding_mode="border", align_corners=False)

    # Keep heatmap values in valid range
    plane_aug = torch.clamp(plane_aug, 0.0, 1.0)
    return img_aug[0], plane_aug[0], coord_aug[0]


# Randomly sample a small affine augmentation.
# This improves robustness while keeping anatomical changes modest.
def random_augmentation_affine(device="cpu", apply_flips=False):
    # Small translations in normalised grid coordinates
    tx = random.uniform(-0.05, 0.05)
    ty = random.uniform(-0.05, 0.05)
    tz = random.uniform(-0.03, 0.03)

    # Small anisotropic scaling
    sx = random.uniform(0.97, 1.03)
    sy = random.uniform(0.97, 1.03)
    sz = random.uniform(0.98, 1.02)

    # Small rotations in degrees
    rx = random.uniform(-5.0, 5.0)
    ry = random.uniform(-5.0, 5.0)
    rz = random.uniform(-6.0, 6.0)

    A = make_affine_3d(
        tx=tx, ty=ty, tz=tz,
        sx=sx, sy=sy, sz=sz,
        rx_deg=rx, ry_deg=ry, rz_deg=rz,
        device=device
    )

    # Optional left/right or anterior/posterior style flips
    if apply_flips:
        flip_x = -1.0 if random.random() < 0.5 else 1.0
        flip_y = -1.0 if random.random() < 0.5 else 1.0
        flip_z = 1.0
        Fm = torch.diag(torch.tensor([flip_x, flip_y, flip_z], dtype=torch.float32, device=device))
        A[:, :3] = A[:, :3] @ Fm

    return A

In [ ]:
# =========================
# DATASET
# =========================

# PyTorch Dataset for the clinical LGE-to-plane heatmap task.
class RealPlaneDataset(torch.utils.data.Dataset):
    def __init__(self, processed_samples, coord_channels, augment=False, apply_flips=False):
        self.samples = processed_samples
        self.coord_channels = coord_channels
        self.augment = augment
        self.apply_flips = apply_flips

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # Clone tensors so augmentations do not modify stored data
        X_img = s["X_img"].clone()
        Y_plane = s["Y_plane"].clone()
        coord = self.coord_channels.clone()

        # Optional paired augmentation of image, heatmap, and coordinate channels
        if self.augment:
            A = random_augmentation_affine(device="cpu", apply_flips=self.apply_flips)
            X_img, Y_plane, coord = apply_paired_affine_3d(X_img, Y_plane, coord, A)

        # Re-estimate the GT normal after augmentation,
        # because augmentation changes the plane orientation.
        n_gt, _ = estimate_plane_from_heatmap(Y_plane[0], topk=TOPK_ESTIMATE)

        # Network input has 4 channels:
        # 1 image channel + 3 coordinate channels
        X = torch.cat([X_img, coord], dim=0)

        # Return all image, target, geometry, and metadata needed for training/evaluation
        return {
            "X": X,
            "Y_plane": Y_plane,
            "Y_normal": n_gt.float(),
            "Y_point_vox": s["Y_point_vox"].clone().float(),

            "gt_cine_stack": s["gt_cine_stack"].clone().float(),
            "stack_offsets_vox": s["stack_offsets_vox"].clone().float(),

            "cine_rows": int(s["cine_rows"]),
            "cine_cols": int(s["cine_cols"]),
            "cine_ps_row": float(s["cine_ps_row"]),
            "cine_ps_col": float(s["cine_ps_col"]),

            "row_vec_vox": s["row_vec_vox"].clone().float(),
            "col_vec_vox": s["col_vec_vox"].clone().float(),

            "slice_thickness_mm": float(s["slice_thickness_mm"]),
            "normal_step_vox_per_mm": float(s["normal_step_vox_per_mm"]),

            "patient_id": s["patient_id"],
        }

In [ ]:
# =========================
# MODEL
# =========================

# Basic two-layer convolution block used throughout the UNet.
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            # First 3D convolution
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),

            # Second 3D convolution
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


# 3D UNet for predicting a 3D heatmap from clinical LGE input.
# Input channels = 4: image + z/y/x coordinate channels.
# Output channels = 1: predicted plane heatmap logits.
class HeatmapUNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=1, base=12):
        super().__init__()

        # Encoder level 1
        self.down1 = DoubleConv(in_channels, base)
        self.pool1 = nn.MaxPool3d(2)

        # Encoder level 2
        self.down2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool3d(2)

        # Encoder level 3
        self.down3 = DoubleConv(base * 2, base * 4)
        self.pool3 = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base * 4, base * 8)

        # Decoder level 3
        self.up3 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv3 = DoubleConv(base * 8 + base * 4, base * 4)

        # Decoder level 2
        self.up2 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv2 = DoubleConv(base * 4 + base * 2, base * 2)

        # Decoder level 1
        self.up1 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv1 = DoubleConv(base * 2 + base, base)

        # Final 1x1 convolution outputs one heatmap logit channel
        self.out = nn.Conv3d(base, out_channels, 1)

    def forward(self, x):
        # Encoder path
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        # Bottleneck representation
        b = self.bottleneck(p3)

        # Decoder path with skip connection from d3
        u3 = self.up3(b)
        if u3.shape[-3:] != d3.shape[-3:]:
            u3 = F.interpolate(u3, size=d3.shape[-3:], mode="trilinear", align_corners=False)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        # Decoder path with skip connection from d2
        u2 = self.up2(u3)
        if u2.shape[-3:] != d2.shape[-3:]:
            u2 = F.interpolate(u2, size=d2.shape[-3:], mode="trilinear", align_corners=False)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        # Decoder path with skip connection from d1
        u1 = self.up1(u2)
        if u1.shape[-3:] != d1.shape[-3:]:
            u1 = F.interpolate(u1, size=d1.shape[-3:], mode="trilinear", align_corners=False)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        # Return raw logits; sigmoid is applied in the loss/evaluation code
        return self.out(u1)

In [ ]:
# =========================
# LOSS
# =========================

# Compute finite-difference gradients in z, y, and x directions.
# This is used to encourage the predicted heatmap edges/smoothness
# to match the GT heatmap.
def gradient_3d(x):
    dz = x[:, :, 1:] - x[:, :, :-1]
    dy = x[:, :, :, 1:] - x[:, :, :, :-1]
    dx = x[:, :, :, :, 1:] - x[:, :, :, :, :-1]
    return dz, dy, dx


# Soft Dice loss for continuous heatmaps.
# This encourages overlap between predicted and GT plane regions.
def soft_dice_loss(pred, target, eps=1e-6):
    num = 2.0 * (pred * target).sum(dim=(1, 2, 3, 4)) + eps
    den = (pred * pred).sum(dim=(1, 2, 3, 4)) + (target * target).sum(dim=(1, 2, 3, 4)) + eps
    dice = num / den
    return 1.0 - dice.mean()


# ---- helpers for geometry-aware loss ----

# Project a vector onto a plane defined by its normal.
def _project_vector_onto_plane(vec, normal, eps=1e-8):
    normal = normal / (normal.norm() + eps)
    proj = vec - torch.dot(vec, normal) * normal
    nrm = proj.norm()
    if nrm < eps:
        return None
    return proj / nrm


# Build in-plane row/column vectors using the predicted normal
# while preserving cine-like row/column orientation as much as possible.
def _build_inplane_vectors_from_reference(pred_normal, ref_row_vec, ref_col_vec, eps=1e-8):
    n = pred_normal / (pred_normal.norm() + eps)

    row_dir = ref_row_vec / (ref_row_vec.norm() + eps)
    col_dir = ref_col_vec / (ref_col_vec.norm() + eps)

    # Try to project the cine column direction into the predicted plane
    col_proj = _project_vector_onto_plane(col_dir, n, eps=eps)
    if col_proj is None:
        # If that fails, use the projected row direction instead
        row_proj = _project_vector_onto_plane(row_dir, n, eps=eps)
        if row_proj is None:
            raise ValueError("Could not construct in-plane basis.")
        u = torch.cross(row_proj, n, dim=0)
        u = u / (u.norm() + eps)
        v = row_proj
    else:
        u = col_proj / (col_proj.norm() + eps)
        v = torch.cross(n, u, dim=0)
        v = v / (v.norm() + eps)

    # Maintain sign consistency with reference cine row direction
    row_proj = _project_vector_onto_plane(row_dir, n, eps=eps)
    if row_proj is not None and torch.dot(v, row_proj) < 0:
        u = -u
        v = -v

    # Maintain sign consistency with reference cine column direction
    col_proj = _project_vector_onto_plane(col_dir, n, eps=eps)
    if col_proj is not None and torch.dot(u, col_proj) < 0:
        u = -u
        v = -v

    # Preserve original cine pixel spacing in voxel units
    row_step = ref_row_vec.norm()
    col_step = ref_col_vec.norm()

    row_vec = v * row_step
    col_vec = u * col_step
    return row_vec, col_vec


# Sample a 2D plane from a 3D volume using an origin and row/column vectors.
def _sample_plane_from_origin_vectors(volume, origin, row_vec, col_vec, out_h, out_w, align_corners=True):
    device_local = volume.device
    D, H, W = volume.shape

    # Build 2D output coordinate grid
    rr = torch.arange(out_h, device=device_local).float()
    cc = torch.arange(out_w, device=device_local).float()
    rr, cc = torch.meshgrid(rr, cc, indexing="ij")

    # Convert every 2D output pixel into a 3D voxel coordinate
    coords = (
        origin[None, None, :]
        + rr[:, :, None] * row_vec[None, None, :]
        + cc[:, :, None] * col_vec[None, None, :]
    )

    z = coords[..., 0]
    y = coords[..., 1]
    x = coords[..., 2]

    # Convert voxel coordinates into grid_sample normalised coordinates [-1,1]
    x_norm = 2.0 * (x / (W - 1)) - 1.0
    y_norm = 2.0 * (y / (H - 1)) - 1.0
    z_norm = 2.0 * (z / (D - 1)) - 1.0

    # PyTorch expects grid order as (x,y,z)
    grid = torch.stack([x_norm, y_norm, z_norm], dim=-1)

    vol_5d = volume[None, None].float()
    grid_5d = grid[None, None]

    # Sample the volume on the requested plane
    sampled = F.grid_sample(
        vol_5d,
        grid_5d,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=align_corners,
    )
    return sampled[0, 0, 0]


# Build offsets and weights for thick-slice averaging.
def _make_slab_offsets_and_weights(half_thickness_vox, num_samples, profile, device_local):
    if num_samples <= 1 or half_thickness_vox < 1e-8:
        offsets = torch.zeros(1, device=device_local)
        weights = torch.ones(1, device=device_local)
        return offsets, weights

    # Sample equally spaced offsets through the slice thickness
    offsets = torch.linspace(-half_thickness_vox, half_thickness_vox, steps=num_samples, device=device_local)

    # Choose weighting profile across the slab
    if profile == "rect":
        weights = torch.ones_like(offsets)
    elif profile == "triangular":
        weights = 1.0 - torch.abs(offsets) / (half_thickness_vox + 1e-8)
        weights = torch.clamp(weights, min=0.0)
    elif profile == "gaussian":
        sigma = max(half_thickness_vox / 2.0, 1e-6)
        weights = torch.exp(-(offsets ** 2) / (2.0 * sigma ** 2))
    else:
        weights = torch.ones_like(offsets)

    # Normalise weights so the thick slice remains intensity-scaled
    weights = weights / (weights.sum() + 1e-8)
    return offsets, weights


# Extract a thick reformatted slice for the slice-supervision component of the loss.
def _extract_thick_slice_for_loss(
    volume,
    center_point,
    normal,
    ref_row_vec,
    ref_col_vec,
    out_h,
    out_w,
    slice_thickness_mm,
    normal_step_vox_per_mm,
    profile="triangular",
    num_samples=9,
):
    device_local = volume.device

    # Move geometry tensors onto same device as the volume
    center_point = center_point.to(device_local).float()
    normal = normal.to(device_local).float()
    ref_row_vec = ref_row_vec.to(device_local).float()
    ref_col_vec = ref_col_vec.to(device_local).float()

    normal = normal / (normal.norm() + 1e-8)

    # Construct row/column sampling vectors in the predicted/GT plane
    row_vec, col_vec = _build_inplane_vectors_from_reference(normal, ref_row_vec, ref_col_vec)

    # Calculate top-left origin of the output slice canvas
    origin = (
        center_point
        - ((out_h - 1) / 2.0) * row_vec
        - ((out_w - 1) / 2.0) * col_vec
    )

    # Convert physical slice thickness into voxel half-thickness
    half_thickness_vox = 0.5 * float(slice_thickness_mm) * float(normal_step_vox_per_mm)

    # Get slab offsets and averaging weights
    offsets, weights = _make_slab_offsets_and_weights(
        half_thickness_vox=half_thickness_vox,
        num_samples=num_samples,
        profile=profile,
        device_local=device_local,
    )

    # Accumulate weighted samples through the slab
    acc = 0.0
    for off, w in zip(offsets, weights):
        origin_k = origin + off * normal
        sl_k = _sample_plane_from_origin_vectors(
            volume, origin_k, row_vec, col_vec, out_h, out_w, align_corners=True
        )
        acc = acc + w * sl_k

    return acc


# Full training loss for clinical heatmap prediction.
# It combines:
# - voxelwise heatmap terms,
# - heatmap gradient consistency,
# - predicted plane normal agreement,
# - image-space slice supervision.
def heatmap_loss(
    pred_logits,
    gt_heat,
    volume_img,
    gt_normal,
    gt_point,
    row_vec_vox,
    col_vec_vox,
    cine_rows,
    cine_cols,
    slice_thickness_mm,
    normal_step_vox_per_mm,
    w_bce=W_BCE,
    w_mse=W_MSE,
    w_dice=W_DICE,
    w_grad=W_GRAD,
    w_normal=W_NORMAL,
    w_slice=W_SLICE,
):
    # Convert logits to probability heatmap
    pred = torch.sigmoid(pred_logits)

    # voxelwise heatmap terms
    bce = F.binary_cross_entropy_with_logits(pred_logits, gt_heat)
    mse = F.mse_loss(pred, gt_heat)
    dice = soft_dice_loss(pred, gt_heat)

    # Gradient consistency between predicted and GT heatmaps
    dz_p, dy_p, dx_p = gradient_3d(pred)
    dz_g, dy_g, dx_g = gradient_3d(gt_heat)

    grad = (
        F.l1_loss(dz_p, dz_g) +
        F.l1_loss(dy_p, dy_g) +
        F.l1_loss(dx_p, dx_g)
    ) / 3.0

    # Weighted voxel-level loss
    voxel_total = w_bce * bce + w_mse * mse + w_dice * dice + w_grad * grad

    # geometry-aware normal term + slice term
    B = pred.shape[0]
    normal_losses = []
    slice_losses = []

    for i in range(B):
        pred_heat_i = pred[i, 0]
        vol_i = volume_img[i, 0]

        gt_normal_i = gt_normal[i]
        gt_point_i = gt_point[i]

        row_vec_i = row_vec_vox[i]
        col_vec_i = col_vec_vox[i]

        # Convert metadata values safely into Python numbers
        cine_rows_i = int(cine_rows[i].item()) if torch.is_tensor(cine_rows[i]) else int(cine_rows[i])
        cine_cols_i = int(cine_cols[i].item()) if torch.is_tensor(cine_cols[i]) else int(cine_cols[i])

        slice_thickness_mm_i = float(slice_thickness_mm[i].item()) if torch.is_tensor(slice_thickness_mm[i]) else float(slice_thickness_mm[i])
        normal_step_vox_per_mm_i = float(normal_step_vox_per_mm[i].item()) if torch.is_tensor(normal_step_vox_per_mm[i]) else float(normal_step_vox_per_mm[i])

        # Estimate predicted plane normal from predicted heatmap
        pred_normal_i, _ = estimate_plane_from_heatmap(pred_heat_i, topk=None)

        # Normal loss is sign-invariant because n and -n define the same plane
        cos_abs = torch.clamp(torch.abs(torch.dot(
            pred_normal_i / (pred_normal_i.norm() + 1e-8),
            gt_normal_i / (gt_normal_i.norm() + 1e-8)
        )), 0.0, 1.0)
        normal_loss_i = 1.0 - cos_abs
        normal_losses.append(normal_loss_i)

        # downsample cine canvas for cheaper training-time slice supervision
        row_scale = (cine_rows_i - 1) / max(LOSS_SLICE_OUT_H - 1, 1)
        col_scale = (cine_cols_i - 1) / max(LOSS_SLICE_OUT_W - 1, 1)

        row_vec_loss = row_vec_i * row_scale
        col_vec_loss = col_vec_i * col_scale

        # GT reformatted slice using GT normal
        slice_gt_i = _extract_thick_slice_for_loss(
            volume=vol_i,
            center_point=gt_point_i,
            normal=gt_normal_i,
            ref_row_vec=row_vec_loss,
            ref_col_vec=col_vec_loss,
            out_h=LOSS_SLICE_OUT_H,
            out_w=LOSS_SLICE_OUT_W,
            slice_thickness_mm=slice_thickness_mm_i,
            normal_step_vox_per_mm=normal_step_vox_per_mm_i,
            profile=REFORMAT_SLICE_PROFILE,
            num_samples=REFORMAT_NUM_SLAB_SAMPLES,
        )

        # Predicted reformatted slice using predicted normal
        slice_pred_i = _extract_thick_slice_for_loss(
            volume=vol_i,
            center_point=gt_point_i,
            normal=pred_normal_i,
            ref_row_vec=row_vec_loss,
            ref_col_vec=col_vec_loss,
            out_h=LOSS_SLICE_OUT_H,
            out_w=LOSS_SLICE_OUT_W,
            slice_thickness_mm=slice_thickness_mm_i,
            normal_step_vox_per_mm=normal_step_vox_per_mm_i,
            profile=REFORMAT_SLICE_PROFILE,
            num_samples=REFORMAT_NUM_SLAB_SAMPLES,
        )

        # Slice loss compares the reformatted GT-normal slice and predicted-normal slice
        slice_rmse_i = torch.sqrt(F.mse_loss(slice_pred_i, slice_gt_i) + 1e-8)
        slice_losses.append(slice_rmse_i)

    # Average geometry terms over batch
    normal_term = torch.stack(normal_losses).mean()
    slice_term = torch.stack(slice_losses).mean()

    # Final combined loss
    total = voxel_total + w_normal * normal_term + w_slice * slice_term
    return total

In [ ]:
# =========================
# TRAINING HELPERS
# =========================
# This cell defines reusable helper functions for creating datasets/loaders
# and for training one model run. It is used both during cross-validation
# and during the later final training stage.

def make_loaders(train_samples_local, val_samples_local, augment_train=True):
    # Create the training dataset.
    # Training can optionally use augmentation to improve robustness.
    train_ds_local = RealPlaneDataset(train_samples_local, coord_channels_fixed, augment=augment_train, apply_flips=APPLY_FLIPS)

    # Create the validation dataset.
    # Validation is never augmented so that validation loss is measured
    # on the true processed patient data.
    val_ds_local   = RealPlaneDataset(val_samples_local, coord_channels_fixed, augment=False)

    # DataLoader for training.
    # shuffle=True randomises the sample order each epoch.
    train_loader_local = torch.utils.data.DataLoader(train_ds_local, batch_size=BATCH_SIZE, shuffle=True)

    # DataLoader for validation.
    # batch_size=1 is used because the clinical samples may be memory-heavy,
    # and because patient-level evaluation is easier one case at a time.
    val_loader_local   = torch.utils.data.DataLoader(val_ds_local, batch_size=1, shuffle=False)

    return train_ds_local, val_ds_local, train_loader_local, val_loader_local


def train_model_once(
    train_samples_local,
    val_samples_local,
    base=12,
    lr=8e-4,
    max_epochs=100,
    patience=18,
    augment_train=True,
):
    # Build datasets and dataloaders for this particular training run.
    _, _, train_loader_local, val_loader_local = make_loaders(
        train_samples_local, val_samples_local, augment_train=augment_train
    )

    # Initialise a fresh 3D U-Net model.
    # The base parameter controls the number of feature channels and therefore model capacity.
    model = HeatmapUNet3D(in_channels=4, out_channels=1, base=base).to(device)

    # AdamW optimiser is used with weight decay for regularisation.
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    # ReduceLROnPlateau lowers the learning rate if the validation loss stops improving.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=max(3, patience // 2)
    )

    # Store the training and validation losses for plotting/inspection.
    train_curve_local = []
    val_curve_local = []

    # Track the best validation result and the corresponding model weights.
    best_val_local = float("inf")
    best_state_local = None
    epochs_no_improve_local = 0

    # Main training loop.
    for epoch in range(1, max_epochs + 1):
        model.train()
        tr_loss = 0.0

        # Loop through each training batch.
        for batch in train_loader_local:
            # Input has 4 channels: image intensity plus three coordinate channels.
            Xb = batch["X"].to(device)

            # Ground-truth plane heatmap.
            Yb = batch["Y_plane"].to(device)

            # The first input channel is the image volume itself.
            # This is used for the geometry-aware slice loss.
            Vb = Xb[:, 0:1]

            # Ground-truth plane normal and fixed centre point.
            Nb = batch["Y_normal"].to(device)
            Pb = batch["Y_point_vox"].to(device)

            # Cine geometry information used by the slice-based loss term.
            row_vec_b = batch["row_vec_vox"].to(device)
            col_vec_b = batch["col_vec_vox"].to(device)
            cine_rows_b = batch["cine_rows"]
            cine_cols_b = batch["cine_cols"]
            slice_thickness_b = batch["slice_thickness_mm"]
            normal_step_b = batch["normal_step_vox_per_mm"]

            # Clear old gradients.
            opt.zero_grad()

            # Forward pass through the model to predict heatmap logits.
            logits = model(Xb)

            # Compute the combined heatmap, normal, and slice-supervision loss.
            loss = heatmap_loss(
                pred_logits=logits,
                gt_heat=Yb,
                volume_img=Vb,
                gt_normal=Nb,
                gt_point=Pb,
                row_vec_vox=row_vec_b,
                col_vec_vox=col_vec_b,
                cine_rows=cine_rows_b,
                cine_cols=cine_cols_b,
                slice_thickness_mm=slice_thickness_b,
                normal_step_vox_per_mm=normal_step_b,
            )

            # Backpropagate gradients and update model weights.
            loss.backward()
            opt.step()

            # Accumulate the batch loss.
            tr_loss += float(loss.detach().cpu())

        # Average training loss over all training batches.
        tr_loss /= len(train_loader_local)
        train_curve_local.append(tr_loss)

        # Validation phase.
        model.eval()
        va_loss = 0.0

        # Disable gradient computation during validation.
        with torch.no_grad():
            for batch in val_loader_local:
                Xb = batch["X"].to(device)
                Yb = batch["Y_plane"].to(device)

                Vb = Xb[:, 0:1]
                Nb = batch["Y_normal"].to(device)
                Pb = batch["Y_point_vox"].to(device)

                row_vec_b = batch["row_vec_vox"].to(device)
                col_vec_b = batch["col_vec_vox"].to(device)
                cine_rows_b = batch["cine_rows"]
                cine_cols_b = batch["cine_cols"]
                slice_thickness_b = batch["slice_thickness_mm"]
                normal_step_b = batch["normal_step_vox_per_mm"]

                # Predict validation heatmap logits.
                logits = model(Xb)

                # Compute validation loss using the same combined loss function.
                loss = heatmap_loss(
                    pred_logits=logits,
                    gt_heat=Yb,
                    volume_img=Vb,
                    gt_normal=Nb,
                    gt_point=Pb,
                    row_vec_vox=row_vec_b,
                    col_vec_vox=col_vec_b,
                    cine_rows=cine_rows_b,
                    cine_cols=cine_cols_b,
                    slice_thickness_mm=slice_thickness_b,
                    normal_step_vox_per_mm=normal_step_b,
                )

                va_loss += float(loss.detach().cpu())

        # Average validation loss over validation batches.
        va_loss /= len(val_loader_local)
        val_curve_local.append(va_loss)

        # Update the learning-rate scheduler based on validation loss.
        scheduler.step(va_loss)

        # Save the best model weights if validation loss improves.
        if va_loss < best_val_local - 1e-4:
            best_val_local = va_loss
            best_state_local = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve_local = 0
        else:
            # Count epochs without sufficient validation improvement.
            epochs_no_improve_local += 1

        # Stop training early if the validation loss has plateaued.
        if epochs_no_improve_local >= patience:
            break

    # Restore the best model weights before returning.
    model.load_state_dict(best_state_local)
    model.to(device)

    return model, best_val_local, train_curve_local, val_curve_local

In [ ]:
# =========================
# FAST CV
# =========================
# This cell implements a lightweight cross-validation procedure.
# It tests a small set of model/lr configurations and chooses the one
# with the lowest mean validation loss across folds.

def make_folds(samples_local, n_folds=2, seed=0):
    # Create shuffled sample indices.
    idx = list(range(len(samples_local)))
    rng = random.Random(seed)
    rng.shuffle(idx)

    # Split indices into approximately equal folds.
    folds = np.array_split(idx, n_folds)
    return [list(f) for f in folds]


def run_cv(trainval_samples_local, cv_configs, n_folds=2, seed=0):
    # Create cross-validation folds from the combined train+validation samples.
    folds = make_folds(trainval_samples_local, n_folds=n_folds, seed=seed)

    results = []

    # Try each candidate hyperparameter configuration.
    for cfg in cv_configs:
        print("\n" + "=" * 70)
        print("CV config:", cfg)
        print("=" * 70)

        fold_vals = []

        # Loop over folds, using one fold for validation and the rest for training.
        for f in range(n_folds):
            val_idx_local = folds[f]
            train_idx_local = []

            # Combine all folds except the current validation fold into the training set.
            for j in range(n_folds):
                if j != f:
                    train_idx_local.extend(folds[j])

            train_subset = [trainval_samples_local[i] for i in train_idx_local]
            val_subset = [trainval_samples_local[i] for i in val_idx_local]

            # Train a model for this fold and configuration.
            _, best_val_local, _, _ = train_model_once(
                train_subset,
                val_subset,
                base=cfg["base"],
                lr=cfg["lr"],
                max_epochs=CV_EPOCHS,
                patience=CV_PATIENCE,
                augment_train=True,
            )

            # Store the best validation loss for this fold.
            fold_vals.append(best_val_local)
            print(f"Fold {f+1}/{n_folds} best val: {best_val_local:.6f}")

        # Mean validation loss across folds is used to compare configurations.
        mean_val = float(np.mean(fold_vals))
        print("Mean CV val:", mean_val)

        results.append({
            "config": cfg,
            "fold_vals": fold_vals,
            "mean_val": mean_val,
        })

    # Sort configurations by mean validation loss, lowest first.
    results = sorted(results, key=lambda x: x["mean_val"])

    # Select the best-performing configuration.
    best_cfg = results[0]["config"]
    return best_cfg, results

In [ ]:
# =========================
# RUN FAST CV
# =========================
# This cell runs the cross-validation search if DO_CV is enabled.
# Otherwise, it simply uses the first configuration in CV_CONFIGS.

if DO_CV:
    # Run fast cross-validation on the train+validation set.
    best_cfg, cv_results = run_cv(trainval_samples, CV_CONFIGS, n_folds=N_FOLDS, seed=SEED)

    # Print the chosen hyperparameters.
    print("\nChosen best config:", best_cfg)

    # Print all CV results for transparency.
    print("\nAll CV results:")
    for r in cv_results:
        print(r)
else:
    # If cross-validation is disabled, use the first predefined configuration.
    best_cfg = CV_CONFIGS[0]
    print("Skipping CV. Using:", best_cfg)

In [ ]:
# =========================
# FINAL TRAINING
# =========================
# This cell trains the final model using the selected hyperparameters.
# It uses the original train/validation/test split and saves the best
# model state according to validation loss.

# Build datasets for final training, validation, and testing.
train_ds = RealPlaneDataset(train_samples, coord_channels_fixed, augment=AUGMENT_TRAIN, apply_flips=APPLY_FLIPS)
val_ds   = RealPlaneDataset(val_samples,   coord_channels_fixed, augment=False)
test_ds  = RealPlaneDataset(test_samples,  coord_channels_fixed, augment=False)

# Create dataloaders.
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False)

# Initialise the final model using the best base-channel size from CV.
model = HeatmapUNet3D(in_channels=4, out_channels=1, base=best_cfg["base"]).to(device)

# Optimiser and learning-rate scheduler for final training.
opt = torch.optim.AdamW(model.parameters(), lr=best_cfg["lr"], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=8)

# Store loss curves for plotting.
train_curve = []
val_curve = []

# Track the best validation model.
best_val = float("inf")
best_state = None
epochs_no_improve = 0

# Main final training loop.
for epoch in range(1, NUM_EPOCHS_FINAL + 1):
    model.train()
    tr_loss = 0.0

    # Training pass.
    for batch in train_loader:
        Xb = batch["X"].to(device)
        Yb = batch["Y_plane"].to(device)

        # Extract volume image channel for geometry-aware loss.
        Vb = Xb[:, 0:1]

        # Ground-truth plane normal and point.
        Nb = batch["Y_normal"].to(device)
        Pb = batch["Y_point_vox"].to(device)

        # Cine geometry metadata for slice supervision.
        row_vec_b = batch["row_vec_vox"].to(device)
        col_vec_b = batch["col_vec_vox"].to(device)
        cine_rows_b = batch["cine_rows"]
        cine_cols_b = batch["cine_cols"]
        slice_thickness_b = batch["slice_thickness_mm"]
        normal_step_b = batch["normal_step_vox_per_mm"]

        # Reset gradients.
        opt.zero_grad()

        # Forward pass.
        logits = model(Xb)

        # Compute combined loss.
        loss = heatmap_loss(
            pred_logits=logits,
            gt_heat=Yb,
            volume_img=Vb,
            gt_normal=Nb,
            gt_point=Pb,
            row_vec_vox=row_vec_b,
            col_vec_vox=col_vec_b,
            cine_rows=cine_rows_b,
            cine_cols=cine_cols_b,
            slice_thickness_mm=slice_thickness_b,
            normal_step_vox_per_mm=normal_step_b,
        )

        # Backpropagation and optimiser update.
        loss.backward()
        opt.step()

        tr_loss += float(loss.detach().cpu())

    # Average training loss for this epoch.
    tr_loss /= len(train_loader)
    train_curve.append(tr_loss)

    # Validation pass.
    model.eval()
    va_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            Xb = batch["X"].to(device)
            Yb = batch["Y_plane"].to(device)

            Vb = Xb[:, 0:1]
            Nb = batch["Y_normal"].to(device)
            Pb = batch["Y_point_vox"].to(device)

            row_vec_b = batch["row_vec_vox"].to(device)
            col_vec_b = batch["col_vec_vox"].to(device)
            cine_rows_b = batch["cine_rows"]
            cine_cols_b = batch["cine_cols"]
            slice_thickness_b = batch["slice_thickness_mm"]
            normal_step_b = batch["normal_step_vox_per_mm"]

            # Predict validation heatmap.
            logits = model(Xb)

            # Compute validation loss.
            loss = heatmap_loss(
                pred_logits=logits,
                gt_heat=Yb,
                volume_img=Vb,
                gt_normal=Nb,
                gt_point=Pb,
                row_vec_vox=row_vec_b,
                col_vec_vox=col_vec_b,
                cine_rows=cine_rows_b,
                cine_cols=cine_cols_b,
                slice_thickness_mm=slice_thickness_b,
                normal_step_vox_per_mm=normal_step_b,
            )

            va_loss += float(loss.detach().cpu())

    # Average validation loss.
    va_loss /= len(val_loader)
    val_curve.append(va_loss)

    # Adjust learning rate if validation loss plateaus.
    scheduler.step(va_loss)
    current_lr = opt.param_groups[0]["lr"]

    # Save the best model state.
    if va_loss < best_val - 1e-4:
        best_val = va_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    # Print progress every 10 epochs and at the first epoch.
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train {tr_loss:.6f} | Val {va_loss:.6f} | LR {current_lr:.2e}")

    # Stop if validation loss has not improved for the patience window.
    if epochs_no_improve >= PATIENCE_FINAL:
        print(f"Early stopping at epoch {epoch}")
        break

# Restore the best validation model.
model.load_state_dict(best_state)
model.to(device)
print("Best val loss:", best_val)

# -------------------------
# VISUALISATION DATASETS
# non-augmented so angle errors stay consistent
# across heatmap and stack cells
# -------------------------
# These datasets are used only for visualisation and evaluation plots.
# Augmentation is disabled so the displayed predictions correspond to the
# true processed clinical volumes.
train_vis_ds = RealPlaneDataset(train_samples, coord_channels_fixed, augment=False)
test_vis_ds  = RealPlaneDataset(test_samples,  coord_channels_fixed, augment=False)

In [ ]:
# =========================
# PLOT TRAINING CURVE
# =========================
# This cell plots the final training and validation loss curves.
# It helps check convergence and possible overfitting.

plt.figure(figsize=(7, 5))

# Plot training loss over epochs.
plt.plot(train_curve, label="Train")

# Plot validation loss over epochs.
plt.plot(val_curve, label="Val")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Clinical heatmap-only training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# EVALUATION
# =========================
# This cell evaluates the trained model on a dataset.
# It reports heatmap errors and geometric plane errors.

def evaluate_dataset(model, dataset, device="cpu"):
    model.eval()
    rows = []

    # Evaluation does not need gradients.
    with torch.no_grad():
        for idx in range(len(dataset)):
            # Load one patient/sample from the dataset.
            batch = dataset[idx]

            # Add batch dimension and move input to device.
            Xb = batch["X"].unsqueeze(0).to(device)

            # Predict heatmap logits and convert to probabilities.
            logits = model(Xb)
            pred_heat = torch.sigmoid(logits)[0, 0].cpu()

            # Ground-truth heatmap and geometry.
            gt_heat_i = batch["Y_plane"][0].cpu()
            n_gt = batch["Y_normal"].cpu()
            p_gt = batch["Y_point_vox"].cpu()
            pid = batch["patient_id"]

            # Voxelwise heatmap errors.
            l1 = F.l1_loss(pred_heat, gt_heat_i).item()
            mse = F.mse_loss(pred_heat, gt_heat_i).item()

            # Estimate the predicted plane normal from the predicted heatmap.
            n_pr, _ = estimate_plane_from_heatmap(pred_heat, topk=TOPK_ESTIMATE)

            # fixed-centre formulation
            # The model only predicts orientation through the heatmap.
            # The point is fixed at the target volume centre.
            p_pr = p_gt.clone()

            # Geometric errors.
            ang = angle_between(n_gt, n_pr)
            pe = point_error(p_pr, p_gt)
            poe = plane_offset_error(n_gt, p_gt, p_pr)

            # Store per-patient results.
            rows.append({
                "patient_id": pid,
                "l1": l1,
                "mse": mse,
                "angle_error_deg": ang,
                "point_error_vox": pe,
                "plane_offset_error_vox": poe,
            })

    # Convert patient-level results into a DataFrame.
    df = pd.DataFrame(rows)

    # Compute summary metrics across the dataset.
    summary = {
        "L1_mean": float(df["l1"].mean()),
        "MSE_mean": float(df["mse"].mean()),
        "AngleError_mean_deg": float(df["angle_error_deg"].mean()),
        "PointError_mean_vox": float(df["point_error_vox"].mean()),
        "PlaneOffsetError_mean_vox": float(df["plane_offset_error_vox"].mean()),
    }
    return summary, df


# Evaluate train, validation, and test datasets.
train_summary, train_df = evaluate_dataset(model, train_ds, device=device)
val_summary, val_df = evaluate_dataset(model, val_ds, device=device)
test_summary, test_df = evaluate_dataset(model, test_ds, device=device)

# Print summary metrics.
print("TRAIN SUMMARY")
print(train_summary)

print("\nVAL SUMMARY")
print(val_summary)

print("\nTEST SUMMARY")
print(test_summary)

# Display test patients sorted by angle error.
display(test_df.sort_values("angle_error_deg"))

In [ ]:
# =========================
# VISUALISATION HELPERS
# =========================
# This cell defines helper functions for visualising predicted and
# ground-truth heatmaps on axial, coronal, and sagittal views.

def plot_case_heatmaps(vol, gt_heat, pred_heat, patient_id, gt_point, pred_point):
    # Get volume dimensions.
    D, H, W = vol.shape

    # Convert GT and predicted centre points to integer voxel indices.
    z_gt, y_gt, x_gt = [int(round(float(v))) for v in gt_point]
    z_pr, y_pr, x_pr = [int(round(float(v))) for v in pred_point]

    # Clip indices so they remain inside the volume.
    z_gt = int(np.clip(z_gt, 0, D - 1))
    y_gt = int(np.clip(y_gt, 0, H - 1))
    x_gt = int(np.clip(x_gt, 0, W - 1))

    z_pr = int(np.clip(z_pr, 0, D - 1))
    y_pr = int(np.clip(y_pr, 0, H - 1))
    x_pr = int(np.clip(x_pr, 0, W - 1))

    # Window the image volume for clearer display.
    vol_disp = window_tensor(vol, low_pct=WINDOW_PERCENTILES[0], high_pct=WINDOW_PERCENTILES[1])

    plt.figure(figsize=(16, 10))

    # GT axial view.
    plt.subplot(2, 3, 1)
    plt.imshow(vol_disp[z_gt], cmap="gray")
    plt.imshow(gt_heat[z_gt], cmap="hot", alpha=0.55)
    plt.scatter([x_gt], [y_gt], marker='x', s=120)
    plt.title(f"{patient_id}\nGT axial")
    plt.axis("off")

    # GT coronal view.
    plt.subplot(2, 3, 2)
    plt.imshow(vol_disp[:, y_gt, :], cmap="gray", aspect="auto")
    plt.imshow(gt_heat[:, y_gt, :], cmap="hot", alpha=0.55, aspect="auto")
    plt.scatter([x_gt], [z_gt], marker='x', s=120)
    plt.title("GT coronal")
    plt.axis("off")

    # GT sagittal view.
    plt.subplot(2, 3, 3)
    plt.imshow(vol_disp[:, :, x_gt], cmap="gray", aspect="auto")
    plt.imshow(gt_heat[:, :, x_gt], cmap="hot", alpha=0.55, aspect="auto")
    plt.scatter([y_gt], [z_gt], marker='x', s=120)
    plt.title("GT sagittal")
    plt.axis("off")

    # Predicted axial view.
    plt.subplot(2, 3, 4)
    plt.imshow(vol_disp[z_pr], cmap="gray")
    plt.imshow(pred_heat[z_pr], cmap="hot", alpha=0.55)
    plt.scatter([x_pr], [y_pr], marker='x', s=120)
    plt.title("Pred axial")
    plt.axis("off")

    # Predicted coronal view.
    plt.subplot(2, 3, 5)
    plt.imshow(vol_disp[:, y_pr, :], cmap="gray", aspect="auto")
    plt.imshow(pred_heat[:, y_pr, :], cmap="hot", alpha=0.55, aspect="auto")
    plt.scatter([x_pr], [z_pr], marker='x', s=120)
    plt.title("Pred coronal")
    plt.axis("off")

    # Predicted sagittal view.
    plt.subplot(2, 3, 6)
    plt.imshow(vol_disp[:, :, x_pr], cmap="gray", aspect="auto")
    plt.imshow(pred_heat[:, :, x_pr], cmap="hot", alpha=0.55, aspect="auto")
    plt.scatter([y_pr], [z_pr], marker='x', s=120)
    plt.title("Pred sagittal")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


def get_case_prediction(model, batch_i, device=device):
    # Extract input and ground-truth information for one case.
    Xt = batch_i["X"]
    gt_heat = batch_i["Y_plane"][0].cpu()
    gt_normal = batch_i["Y_normal"].cpu()
    gt_point = batch_i["Y_point_vox"].cpu()
    patient_id = batch_i["patient_id"]

    # Predict heatmap for this case.
    with torch.no_grad():
        pred_logits = model(Xt.unsqueeze(0).to(device))
        pred_heat = torch.sigmoid(pred_logits)[0, 0].cpu()

    # Estimate predicted plane normal from the predicted heatmap.
    pred_normal, _ = estimate_plane_from_heatmap(pred_heat, topk=TOPK_ESTIMATE)
    pred_normal = pred_normal.cpu()

    # fixed-centre formulation
    # Predicted point is fixed to the same centre as GT.
    pred_point = gt_point.clone()

    # Compute angular error between predicted and GT plane normals.
    angle_err = angle_between(gt_normal, pred_normal)

    # Extract the original image volume from the first channel.
    vol = Xt[0].cpu()

    return {
        "Xt": Xt,
        "vol": vol,
        "gt_heat": gt_heat,
        "gt_normal": gt_normal,
        "gt_point": gt_point,
        "pred_heat": pred_heat,
        "pred_normal": pred_normal,
        "pred_point": pred_point,
        "angle_err": angle_err,
        "patient_id": patient_id,
    }

In [ ]:
# =========================
# REFORMATTING
# - fixed centre
# - exact cine canvas
# - projected cine in-plane basis
# - thick-slice averaging
# =========================
# This cell contains the functions used to convert a predicted plane normal
# into reformatted 2D LGE slices and full short-axis stacks.

def project_vector_onto_plane(vec, normal, eps=1e-8):
    # Normalise the plane normal.
    normal = normal / (normal.norm() + eps)

    # Remove the component of vec along the normal.
    proj = vec - torch.dot(vec, normal) * normal

    # If the projection is too small, the vector is nearly parallel to the normal.
    nrm = proj.norm()
    if nrm < eps:
        return None

    # Return the projected unit vector.
    return proj / nrm


def build_inplane_vectors_from_reference(pred_normal, ref_row_vec, ref_col_vec, eps=1e-8):
    """
    Preserve cine-like in-plane geometry as much as possible while forcing
    the basis to lie in the predicted plane.
    """
    # Normalise the predicted plane normal.
    n = pred_normal / (pred_normal.norm() + eps)

    # Convert reference row/column vectors into directions.
    row_dir = ref_row_vec / (ref_row_vec.norm() + eps)
    col_dir = ref_col_vec / (ref_col_vec.norm() + eps)

    # Try to project the cine column direction into the predicted plane.
    col_proj = project_vector_onto_plane(col_dir, n, eps=eps)

    if col_proj is None:
        # If the column direction is unusable, fall back to the row direction.
        row_proj = project_vector_onto_plane(row_dir, n, eps=eps)
        if row_proj is None:
            raise ValueError("Could not construct in-plane basis.")

        # Build an orthogonal in-plane basis from row projection and normal.
        u = torch.cross(row_proj, n, dim=0)
        u = u / (u.norm() + eps)
        v = row_proj
    else:
        # Use the projected column direction as one in-plane axis.
        u = col_proj / (col_proj.norm() + eps)

        # Create the second in-plane direction using the plane normal.
        v = torch.cross(n, u, dim=0)
        v = v / (v.norm() + eps)

    # sign consistency with reference cine axes
    # Flip both axes if needed so the row direction matches the original cine orientation.
    row_proj = project_vector_onto_plane(row_dir, n, eps=eps)
    if row_proj is not None and torch.dot(v, row_proj) < 0:
        u = -u
        v = -v

    # Flip both axes if needed so the column direction matches the original cine orientation.
    col_proj = project_vector_onto_plane(col_dir, n, eps=eps)
    if col_proj is not None and torch.dot(u, col_proj) < 0:
        u = -u
        v = -v

    # Preserve original voxel step sizes along cine row and column directions.
    row_step = ref_row_vec.norm()
    col_step = ref_col_vec.norm()

    # Scale unit vectors back to one-pixel displacement vectors.
    row_vec = v * row_step
    col_vec = u * col_step
    return row_vec, col_vec


def sample_plane_from_origin_vectors(volume, origin, row_vec, col_vec, out_h, out_w, align_corners=True):
    # Sample a 2D plane from a 3D volume using an origin and two in-plane vectors.
    device_local = volume.device
    D, H, W = volume.shape

    # Create output pixel coordinate grid.
    rr = torch.arange(out_h, device=device_local).float()
    cc = torch.arange(out_w, device=device_local).float()
    rr, cc = torch.meshgrid(rr, cc, indexing="ij")

    # Convert each output pixel location into a 3D voxel coordinate.
    coords = (
        origin[None, None, :]
        + rr[:, :, None] * row_vec[None, None, :]
        + cc[:, :, None] * col_vec[None, None, :]
    )

    z = coords[..., 0]
    y = coords[..., 1]
    x = coords[..., 2]

    # Convert voxel coordinates to normalised grid_sample coordinates.
    x_norm = 2.0 * (x / (W - 1)) - 1.0
    y_norm = 2.0 * (y / (H - 1)) - 1.0
    z_norm = 2.0 * (z / (D - 1)) - 1.0

    # grid_sample expects order x, y, z in the final grid dimension.
    grid = torch.stack([x_norm, y_norm, z_norm], dim=-1)

    # Add batch and channel dimensions for grid_sample.
    vol_5d = volume[None, None].float()
    grid_5d = grid[None, None]

    # Sample the plane from the 3D volume.
    sampled = F.grid_sample(
        vol_5d,
        grid_5d,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=align_corners,
    )

    # Remove batch/channel/depth dimensions to return a 2D slice.
    return sampled[0, 0, 0]


def make_slab_offsets_and_weights(half_thickness_vox, num_samples, profile, device_local):
    # If no thick-slice averaging is required, use only the central plane.
    if num_samples <= 1 or half_thickness_vox < 1e-8:
        offsets = torch.zeros(1, device=device_local)
        weights = torch.ones(1, device=device_local)
        return offsets, weights

    # Create offsets along the plane normal across the slice thickness.
    offsets = torch.linspace(-half_thickness_vox, half_thickness_vox, steps=num_samples, device=device_local)

    # Rectangular slice profile gives equal weight to each sampled plane.
    if profile == "rect":
        weights = torch.ones_like(offsets)

    # Triangular profile gives highest weight to the centre and lower weight near edges.
    elif profile == "triangular":
        weights = 1.0 - torch.abs(offsets) / (half_thickness_vox + 1e-8)
        weights = torch.clamp(weights, min=0.0)

    # Gaussian profile smoothly weights central samples more strongly.
    elif profile == "gaussian":
        sigma = max(half_thickness_vox / 2.0, 1e-6)
        weights = torch.exp(-(offsets ** 2) / (2.0 * sigma ** 2))

    # Fallback to equal weighting if an unknown profile is supplied.
    else:
        weights = torch.ones_like(offsets)

    # Normalise weights so the weighted sum preserves intensity scale.
    weights = weights / (weights.sum() + 1e-8)
    return offsets, weights


def extract_thick_slice(
    volume,
    center_point,
    normal,
    ref_row_vec,
    ref_col_vec,
    out_h,
    out_w,
    slice_thickness_mm,
    normal_step_vox_per_mm,
    profile=REFORMAT_SLICE_PROFILE,
    num_samples=REFORMAT_NUM_SLAB_SAMPLES,
    align_corners=True,
):
    """
    Fixed-centre thick-slice reformat with exact cine-sized canvas.
    """
    device_local = volume.device

    # Move geometric inputs to the same device as the volume.
    center_point = center_point.to(device_local).float()
    normal = normal.to(device_local).float()
    ref_row_vec = ref_row_vec.to(device_local).float()
    ref_col_vec = ref_col_vec.to(device_local).float()

    # Normalise the plane normal.
    normal = normal / (normal.norm() + 1e-8)

    # Construct cine-like in-plane row and column vectors lying in the predicted plane.
    row_vec, col_vec = build_inplane_vectors_from_reference(normal, ref_row_vec, ref_col_vec)

    # Compute the top-left origin of the output slice canvas.
    origin = (
        center_point
        - ((out_h - 1) / 2.0) * row_vec
        - ((out_w - 1) / 2.0) * col_vec
    )

    # Convert slice thickness in mm into voxel units along the plane normal.
    half_thickness_vox = 0.5 * float(slice_thickness_mm) * float(normal_step_vox_per_mm)

    # Generate slab offsets and weights for thick-slice averaging.
    offsets, weights = make_slab_offsets_and_weights(
        half_thickness_vox=half_thickness_vox,
        num_samples=num_samples,
        profile=profile,
        device_local=device_local,
    )

    # Accumulate weighted samples through the slab.
    acc = 0.0
    for off, w in zip(offsets, weights):
        origin_k = origin + off * normal
        sl_k = sample_plane_from_origin_vectors(
            volume, origin_k, row_vec, col_vec, out_h, out_w, align_corners=align_corners
        )
        acc = acc + w * sl_k

    return acc


def extract_stack_from_offsets(
    volume,
    normal,
    base_point,
    offsets_vox,
    out_h,
    out_w,
    ref_row_vec,
    ref_col_vec,
    slice_thickness_mm,
    normal_step_vox_per_mm,
    profile=REFORMAT_SLICE_PROFILE,
    num_samples=REFORMAT_NUM_SLAB_SAMPLES,
):
    # Extract multiple parallel reformatted slices using known stack offsets.
    slices = []

    # Normalise plane normal.
    normal = normal / (normal.norm() + 1e-8)

    # For each offset, move the centre point along the normal and extract a slice.
    for off in offsets_vox:
        center_k = base_point + off * normal
        sl = extract_thick_slice(
            volume=volume,
            center_point=center_k,
            normal=normal,
            ref_row_vec=ref_row_vec,
            ref_col_vec=ref_col_vec,
            out_h=out_h,
            out_w=out_w,
            slice_thickness_mm=slice_thickness_mm,
            normal_step_vox_per_mm=normal_step_vox_per_mm,
            profile=profile,
            num_samples=num_samples,
            align_corners=True,
        )
        slices.append(sl)

    return slices


def normalize_slice_01(x, eps=1e-8):
    # Min-max normalise a 2D slice to [0, 1] for display and RMSE comparison.
    x = x.float()
    x = x - x.min()
    x = x / (x.max() + eps)
    return x

In [ ]:
# =========================
# SHOW 10 TRAINING CASES
# =========================
# This cell visualises predicted vs ground-truth heatmaps for up to
# 10 training cases.

model.eval()

# Limit the number of displayed cases to 10 or fewer.
n_show_train = min(10, len(train_vis_ds))

for i in range(n_show_train):
    # Load one non-augmented training case.
    batch_i = train_vis_ds[i]

    # Get prediction, GT heatmap, predicted normal, and angle error.
    case = get_case_prediction(model, batch_i, device=device)

    print(f"\nTRAIN CASE {i+1} | {case['patient_id']}")
    print("Angle error (deg):", case["angle_err"])

    # Show GT and predicted heatmaps overlaid on the LGE volume.
    plot_case_heatmaps(
        vol=case["vol"],
        gt_heat=case["gt_heat"],
        pred_heat=case["pred_heat"],
        patient_id=case["patient_id"],
        gt_point=case["gt_point"],
        pred_point=case["pred_point"]
    )

In [ ]:
# =========================
# SHOW 10 TEST CASES
# =========================
# This cell visualises predicted vs ground-truth heatmaps for up to
# 10 held-out test cases.

model.eval()

# Limit the number of displayed test cases to 10 or fewer.
n_show_test = min(10, len(test_vis_ds))

for i in range(n_show_test):
    # Load one non-augmented test case.
    batch_i = test_vis_ds[i]

    # Get model prediction and geometric error.
    case = get_case_prediction(model, batch_i, device=device)

    print(f"\nTEST CASE {i+1} | {case['patient_id']}")
    print("Angle error (deg):", case["angle_err"])

    # Visualise GT and predicted heatmaps in three anatomical views.
    plot_case_heatmaps(
        vol=case["vol"],
        gt_heat=case["gt_heat"],
        pred_heat=case["pred_heat"],
        patient_id=case["patient_id"],
        gt_point=case["gt_point"],
        pred_point=case["pred_point"]
    )

In [ ]:
# =========================
# CELL 25
# TRAIN SET: GT CINE MID-SLICE vs PREDICTED REFORMATTED LGE MID-SLICE
# Metric shown: stack RMSE mean only
# =========================
# This cell compares the true cine mid-slice to the predicted reformatted
# LGE mid-slice for training cases. It also computes mean RMSE across
# the full stack.

model.eval()

train_mid_rows = []

# Show up to 10 training cases.
n_show_train = min(10, len(train_vis_ds))

for i in range(n_show_train):
    # Load one training case and get model prediction.
    batch_i = train_vis_ds[i]
    case = get_case_prediction(model, batch_i, device=device)

    # Ground-truth cine stack and geometry.
    gt_cine_stack = batch_i["gt_cine_stack"]
    gt_point = batch_i["Y_point_vox"]
    stack_offsets = batch_i["stack_offsets_vox"]

    cine_rows = batch_i["cine_rows"]
    cine_cols = batch_i["cine_cols"]

    row_vec_vox = batch_i["row_vec_vox"]
    col_vec_vox = batch_i["col_vec_vox"]

    slice_thickness_mm = batch_i["slice_thickness_mm"]
    normal_step_vox_per_mm = batch_i["normal_step_vox_per_mm"]

    # Extract a full predicted reformatted LGE stack using the predicted normal.
    pred_stack = extract_stack_from_offsets(
        volume=case["vol"],
        normal=case["pred_normal"],
        base_point=gt_point.cpu(),
        offsets_vox=stack_offsets.cpu(),
        out_h=cine_rows,
        out_w=cine_cols,
        ref_row_vec=row_vec_vox.cpu(),
        ref_col_vec=col_vec_vox.cpu(),
        slice_thickness_mm=slice_thickness_mm,
        normal_step_vox_per_mm=normal_step_vox_per_mm,
        profile=REFORMAT_SLICE_PROFILE,
        num_samples=REFORMAT_NUM_SLAB_SAMPLES,
    )

    # Compute RMSE for each slice after normalising both GT and predicted slices.
    stack_rmse = []
    for gt_sl, pred_sl in zip(gt_cine_stack, pred_stack):
        gt_n = normalize_slice_01(gt_sl)
        pred_n = normalize_slice_01(pred_sl)
        rmse_i = float(torch.sqrt(F.mse_loss(pred_n, gt_n)).item())
        stack_rmse.append(rmse_i)

    # Mean RMSE across the stack.
    stack_rmse_mean = float(np.mean(stack_rmse))

    # Select the middle slice for visual comparison.
    mid_idx = len(gt_cine_stack) // 2
    gt_mid_slice = gt_cine_stack[mid_idx]
    pred_mid_slice = pred_stack[mid_idx]

    # Normalise slices for display.
    gt_n = normalize_slice_01(gt_mid_slice)
    pred_n = normalize_slice_01(pred_mid_slice)

    # Store result for this patient.
    train_mid_rows.append({
        "patient_id": case["patient_id"],
        "stack_rmse_mean": stack_rmse_mean,
    })

    print(f"\nTRAIN MID-SLICE CASE {i+1} | {case['patient_id']}")
    print("Stack RMSE mean:", stack_rmse_mean)

    # Display GT cine mid-slice and predicted reformatted LGE mid-slice.
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(gt_n, cmap="gray")
    plt.title("GT cine mid short-axis slice")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(pred_n, cmap="gray")
    plt.title("Predicted reformatted LGE mid-slice")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Display per-patient training mid-slice stack RMSE results.
train_mid_df = pd.DataFrame(train_mid_rows)
display(train_mid_df.sort_values("stack_rmse_mean"))

# Print summary metric.
print("\nTRAIN MID-SLICE SUMMARY")
print({
    "stack_rmse_mean": float(train_mid_df["stack_rmse_mean"].mean()),
})

In [ ]:
# =========================
# CELL 26
# TEST SET: GT CINE MID-SLICE vs PREDICTED REFORMATTED LGE MID-SLICE
# Metric shown: stack RMSE mean only
# =========================
# This cell performs the same mid-slice visual comparison as Cell 25,
# but on the held-out test set.

model.eval()

test_mid_rows = []

# Show up to 10 test cases.
n_show_test = min(10, len(test_vis_ds))

for i in range(n_show_test):
    # Load one test case and get prediction.
    batch_i = test_vis_ds[i]
    case = get_case_prediction(model, batch_i, device=device)

    # Ground-truth cine stack and geometry metadata.
    gt_cine_stack = batch_i["gt_cine_stack"]
    gt_point = batch_i["Y_point_vox"]
    stack_offsets = batch_i["stack_offsets_vox"]

    cine_rows = batch_i["cine_rows"]
    cine_cols = batch_i["cine_cols"]

    row_vec_vox = batch_i["row_vec_vox"]
    col_vec_vox = batch_i["col_vec_vox"]

    slice_thickness_mm = batch_i["slice_thickness_mm"]
    normal_step_vox_per_mm = batch_i["normal_step_vox_per_mm"]

    # Extract predicted reformatted LGE stack from the predicted plane normal.
    pred_stack = extract_stack_from_offsets(
        volume=case["vol"],
        normal=case["pred_normal"],
        base_point=gt_point.cpu(),
        offsets_vox=stack_offsets.cpu(),
        out_h=cine_rows,
        out_w=cine_cols,
        ref_row_vec=row_vec_vox.cpu(),
        ref_col_vec=col_vec_vox.cpu(),
        slice_thickness_mm=slice_thickness_mm,
        normal_step_vox_per_mm=normal_step_vox_per_mm,
        profile=REFORMAT_SLICE_PROFILE,
        num_samples=REFORMAT_NUM_SLAB_SAMPLES,
    )

    # Compute slice-wise RMSE over the stack.
    stack_rmse = []
    for gt_sl, pred_sl in zip(gt_cine_stack, pred_stack):
        gt_n = normalize_slice_01(gt_sl)
        pred_n = normalize_slice_01(pred_sl)
        rmse_i = float(torch.sqrt(F.mse_loss(pred_n, gt_n)).item())
        stack_rmse.append(rmse_i)

    # Average stack RMSE.
    stack_rmse_mean = float(np.mean(stack_rmse))

    # Select middle slice for display.
    mid_idx = len(gt_cine_stack) // 2
    gt_mid_slice = gt_cine_stack[mid_idx]
    pred_mid_slice = pred_stack[mid_idx]

    # Normalise for display.
    gt_n = normalize_slice_01(gt_mid_slice)
    pred_n = normalize_slice_01(pred_mid_slice)

    # Store patient-level result.
    test_mid_rows.append({
        "patient_id": case["patient_id"],
        "stack_rmse_mean": stack_rmse_mean,
    })

    print(f"\nTEST MID-SLICE CASE {i+1} | {case['patient_id']}")
    print("Stack RMSE mean:", stack_rmse_mean)

    # Display GT cine mid-slice against predicted LGE reformat.
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(gt_n, cmap="gray")
    plt.title("GT cine mid short-axis slice")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(pred_n, cmap="gray")
    plt.title("Predicted reformatted LGE mid-slice")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Display per-patient test results.
test_mid_df = pd.DataFrame(test_mid_rows)
display(test_mid_df.sort_values("stack_rmse_mean"))

# Print test summary.
print("\nTEST MID-SLICE SUMMARY")
print({
    "stack_rmse_mean": float(test_mid_df["stack_rmse_mean"].mean()),
})

In [ ]:
# =========================
# CELL 27
# TRAIN SET: GT CINE STACK vs PREDICTED REFORMATTED LGE STACK
# Metric: mean RMSE across stack
# =========================
# This cell visualises the full stack comparison for training cases.
# The top rows show GT cine slices and the lower rows show predicted
# reformatted LGE slices.

def show_stack_comparison(gt_cine_stack, pred_stack, n_cols=9):
    # Number of slices in the stack.
    n_slices = len(gt_cine_stack)

    # Number of rows needed to show all GT slices.
    n_gt_rows = int(np.ceil(n_slices / n_cols))

    # Total rows include GT rows plus predicted rows.
    total_rows = 2 * n_gt_rows

    # Create figure grid.
    fig, axes = plt.subplots(
        total_rows,
        n_cols,
        figsize=(2.2 * n_cols, 2.2 * total_rows)
    )

    # Ensure axes is always 2D.
    if total_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    # Turn off all axes before filling them.
    for ax in axes.reshape(-1):
        ax.axis("off")

    # Fill the grid with GT and predicted slices.
    for i in range(n_slices):
        row = i // n_cols
        col = i % n_cols

        # GT cine slice.
        axes[row, col].imshow(normalize_slice_01(gt_cine_stack[i]), cmap="gray")
        axes[row, col].set_title(f"GT {i}", fontsize=9)
        axes[row, col].axis("off")

        # Corresponding predicted reformatted slice.
        pred_row = row + n_gt_rows
        axes[pred_row, col].imshow(normalize_slice_01(pred_stack[i]), cmap="gray")
        axes[pred_row, col].set_title(f"Pred {i}", fontsize=9)
        axes[pred_row, col].axis("off")

    plt.tight_layout()
    plt.show()


model.eval()

train_stack_rows = []

# Show up to 10 training cases.
n_show_train = min(10, len(train_vis_ds))

for i in range(n_show_train):
    # Load one training case and predict its plane.
    batch_i = train_vis_ds[i]
    out = get_case_prediction(model, batch_i)

    Xt = batch_i["X"]
    gt_cine_stack = batch_i["gt_cine_stack"]
    gt_point = batch_i["Y_point_vox"]
    stack_offsets = batch_i["stack_offsets_vox"]

    cine_rows = batch_i["cine_rows"]
    cine_cols = batch_i["cine_cols"]

    row_vec_vox = batch_i["row_vec_vox"]
    col_vec_vox = batch_i["col_vec_vox"]

    slice_thickness_mm = batch_i["slice_thickness_mm"]
    normal_step_vox_per_mm = batch_i["normal_step_vox_per_mm"]

    patient_id = batch_i["patient_id"]

    # Extract image volume and predicted normal.
    vol = Xt[0].cpu()
    pred_normal = out["pred_normal"]

    # Generate the predicted reformatted LGE stack.
    pred_stack = extract_stack_from_offsets(
        volume=vol,
        normal=pred_normal.cpu(),
        base_point=gt_point.cpu(),
        offsets_vox=stack_offsets.cpu(),
        out_h=cine_rows,
        out_w=cine_cols,
        ref_row_vec=row_vec_vox.cpu(),
        ref_col_vec=col_vec_vox.cpu(),
        slice_thickness_mm=slice_thickness_mm,
        normal_step_vox_per_mm=normal_step_vox_per_mm,
        profile=REFORMAT_SLICE_PROFILE,
        num_samples=REFORMAT_NUM_SLAB_SAMPLES,
    )

    # Compute RMSE for each matched GT/predicted slice.
    stack_rmse = []
    for gt_sl, pred_sl in zip(gt_cine_stack, pred_stack):
        gt_n = normalize_slice_01(gt_sl)
        pred_n = normalize_slice_01(pred_sl)
        rmse_i = float(torch.sqrt(F.mse_loss(pred_n, gt_n)).item())
        stack_rmse.append(rmse_i)

    # Mean stack RMSE for this patient.
    stack_rmse_mean = float(np.mean(stack_rmse))

    # Store patient-level result.
    train_stack_rows.append({
        "patient_id": patient_id,
        "stack_rmse_mean": stack_rmse_mean,
    })

    print(f"\nTRAIN STACK CASE {i+1} | {patient_id}")
    print("Stack RMSE mean:", stack_rmse_mean)

    # Show complete GT vs predicted stack.
    show_stack_comparison(gt_cine_stack, pred_stack, n_cols=9)

# Display training stack results.
train_stack_df = pd.DataFrame(train_stack_rows)
display(train_stack_df.sort_values("stack_rmse_mean"))

# Print summary.
print("\nTRAIN STACK SUMMARY")
print({
    "stack_rmse_mean": float(train_stack_df["stack_rmse_mean"].mean()),
})

In [ ]:
# =========================
# CELL 28
# TEST SET: GT CINE STACK vs PREDICTED REFORMATTED LGE STACK
# Metric shown: stack RMSE mean only
# Layout: 9 slices in first row, remaining slices in second row
# for GT; same again underneath for Pred
# =========================
# This final cell visualises and evaluates full stack reformatting on
# every test case.

model.eval()

test_stack_rows = []

# Loop through all test visualisation cases.
for i in range(len(test_vis_ds)):
    # Load one test case and get predicted plane information.
    batch_i = test_vis_ds[i]
    out = get_case_prediction(model, batch_i)
    case = out

    Xt = batch_i["X"]
    gt_cine_stack = batch_i["gt_cine_stack"]
    gt_point = batch_i["Y_point_vox"]
    stack_offsets = batch_i["stack_offsets_vox"]

    cine_rows = batch_i["cine_rows"]
    cine_cols = batch_i["cine_cols"]

    row_vec_vox = batch_i["row_vec_vox"]
    col_vec_vox = batch_i["col_vec_vox"]

    slice_thickness_mm = batch_i["slice_thickness_mm"]
    normal_step_vox_per_mm = batch_i["normal_step_vox_per_mm"]

    # Extract the predicted reformatted LGE stack using the predicted plane normal.
    pred_stack = extract_stack_from_offsets(
        volume=case["vol"],
        normal=case["pred_normal"],
        base_point=gt_point.cpu(),
        offsets_vox=stack_offsets.cpu(),
        out_h=cine_rows,
        out_w=cine_cols,
        ref_row_vec=row_vec_vox.cpu(),
        ref_col_vec=col_vec_vox.cpu(),
        slice_thickness_mm=slice_thickness_mm,
        normal_step_vox_per_mm=normal_step_vox_per_mm,
        profile=REFORMAT_SLICE_PROFILE,
        num_samples=REFORMAT_NUM_SLAB_SAMPLES,
    )

    # Compute RMSE for each slice in the stack.
    stack_rmse = []
    for gt_sl, pred_sl in zip(gt_cine_stack, pred_stack):
        gt_n = normalize_slice_01(gt_sl)
        pred_n = normalize_slice_01(pred_sl)
        rmse_i = float(torch.sqrt(F.mse_loss(pred_n, gt_n)).item())
        stack_rmse.append(rmse_i)

    # Mean RMSE across the whole stack.
    stack_rmse_mean = float(np.mean(stack_rmse))

    # Store test result for this patient.
    test_stack_rows.append({
        "patient_id": case["patient_id"],
        "stack_rmse_mean": stack_rmse_mean,
    })

    print(f"\nTEST STACK CASE {i+1} | {case['patient_id']}")
    print("Stack RMSE mean:", stack_rmse_mean)

    # Visualise the complete GT cine stack and predicted reformatted LGE stack.
    show_stack_comparison(
    gt_cine_stack=gt_cine_stack,
    pred_stack=pred_stack
)

# Display per-patient test stack RMSE results.
test_stack_df = pd.DataFrame(test_stack_rows)
display(test_stack_df.sort_values("stack_rmse_mean"))

# Print final test stack summary.
print("\nTEST STACK SUMMARY")
print({
    "stack_rmse_mean": float(test_stack_df["stack_rmse_mean"].mean()),
})